# Twitter Sentiment Analysis Using RNN

Week 31 Graded Mini Project  
Advanced Certificate Programme in Applied Artificial Intelligence and Machine Learning

## Objective

Build an end-to-end sentiment analysis pipeline for Twitter posts using a Recurrent Neural Network (RNN). The final model should classify tweets into one of three sentiment classes: `Negative`, `Neutral`, or `Positive`.

## Assignment And Rubric Checklist

This notebook is organized to match the project task list and rubric:

1. Load the dataset.
2. Clean and preprocess tweet text.
3. Engineer numerical text features and token sequences.
4. Conduct exploratory data analysis.
5. Build an embedding-based LSTM or GRU model.
6. Train, validate, and test the model.
7. Evaluate accuracy, precision, recall, and F1-score.
8. Plot learning curves and confusion matrix.
9. Improve the model through controlled hyperparameter experiments.
10. Present final findings and sample predictions.

## Dataset Notes

The source CSV file has no header row. It will be loaded with these explicit column names:

- `tweet_id`
- `entity`
- `sentiment`
- `tweet_text`

Initial profiling found four sentiment labels in the raw dataset: `Negative`, `Positive`, `Neutral`, and `Irrelevant`. Because the assignment objective and rubric describe a three-class sentiment task, the main modeling pipeline will document and exclude `Irrelevant`, then train on `Negative`, `Neutral`, and `Positive`.

## Phase 1: Project Setup

Status: scaffold created.

The implementation will use a reproducible project structure:

- `requirements.txt` for dependencies.
- `PROJECT_CONTEXT.md` for assignment, rubric, and dataset decisions.
- `outputs/data` for cleaned working datasets.
- `outputs/figures` for generated charts.
- `outputs/models` for trained model files.
- `outputs/tables` for evaluation tables and intermediate summaries.
- `outputs/reports` for report or presentation-ready exports.

In [ ]:
# Phase 1 setup cell. Execute this before later phases.
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "twitter_training.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"
REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [FIGURES_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path exists: {DATA_PATH.exists()} -> {DATA_PATH}")

---

# Part 1: Data Processing

## Phase 2: Loading And Raw Data Validation

Phase 2 loads the raw CSV exactly as supplied and validates its structure before any cleaning is performed. This gives the project an auditable baseline for the rubric item `Loading the Dataset` and prepares data quality evidence for the next phase.

Work completed in this phase:

- Load `twitter_training.csv` into a pandas DataFrame with explicit column names.
- Confirm shape, columns, data types, and sample rows.
- Check missing values, blank tweet text, duplicate rows, and sentiment labels.
- Compute raw sentiment distribution and text length statistics.
- Save validation summaries to `outputs/tables` for documentation and later reporting.

No cleaning is applied in this phase. Cleaning starts in Phase 3.

In [ ]:
# Phase 2 imports and constants.
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 20)

RAW_COLUMNS = ["tweet_id", "entity", "sentiment", "tweet_text"]
ASSIGNMENT_SENTIMENTS = {"Negative", "Neutral", "Positive"}

print("Phase 2 constants ready.")
print(f"Expected raw columns: {RAW_COLUMNS}")

In [ ]:
# Load the no-header CSV using explicit column names.
df_raw = pd.read_csv(
    DATA_PATH,
    header=None,
    names=RAW_COLUMNS,
    encoding="utf-8",
)

print(f"Dataset loaded successfully: {DATA_PATH.name}")
print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
print("Columns:", list(df_raw.columns))

display(df_raw.head())

In [ ]:
# Structural validation: schema, data types, missing values, and unique counts.
schema_is_valid = list(df_raw.columns) == RAW_COLUMNS

dataset_overview = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "schema_matches_expected_columns",
            "unique_tweet_ids",
            "unique_entities",
            "unique_sentiments",
            "missing_tweet_text",
            "exact_duplicate_rows",
        ],
        "value": [
            len(df_raw),
            df_raw.shape[1],
            schema_is_valid,
            df_raw["tweet_id"].nunique(dropna=True),
            df_raw["entity"].nunique(dropna=True),
            df_raw["sentiment"].nunique(dropna=True),
            df_raw["tweet_text"].isna().sum(),
            df_raw.duplicated().sum(),
        ],
    }
)

missing_summary = pd.DataFrame(
    {
        "missing_count": df_raw.isna().sum(),
        "missing_percent": (df_raw.isna().mean() * 100).round(2),
        "unique_values": df_raw.nunique(dropna=True),
        "dtype": df_raw.dtypes.astype(str),
    }
).reset_index(names="column")

display(dataset_overview)
display(missing_summary)

In [ ]:
# Sentiment validation.
sentiment_distribution = (
    df_raw["sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="count")
)
sentiment_distribution["percent"] = (
    sentiment_distribution["count"] / len(df_raw) * 100
).round(2)

observed_sentiments = set(df_raw["sentiment"].dropna().unique())
extra_sentiments = sorted(observed_sentiments - ASSIGNMENT_SENTIMENTS)
missing_assignment_sentiments = sorted(ASSIGNMENT_SENTIMENTS - observed_sentiments)

sentiment_validation = pd.DataFrame(
    {
        "check": [
            "assignment_sentiments_present",
            "extra_labels_in_raw_data",
            "recommended_main_model_scope",
        ],
        "result": [
            len(missing_assignment_sentiments) == 0,
            ", ".join(extra_sentiments) if extra_sentiments else "None",
            "Use Negative, Neutral, and Positive; document and exclude Irrelevant before modeling.",
        ],
    }
)

display(sentiment_distribution)
display(sentiment_validation)

In [ ]:
# Duplicate and blank-text validation.
text_as_string = df_raw["tweet_text"].fillna("").astype(str)
blank_or_missing_text = text_as_string.str.strip().eq("")

duplicate_summary = pd.DataFrame(
    {
        "metric": [
            "blank_or_missing_tweet_text_rows",
            "exact_duplicate_rows",
            "duplicate_tweet_text_rows_non_null",
            "duplicate_tweet_text_and_sentiment_rows_non_null",
        ],
        "value": [
            int(blank_or_missing_text.sum()),
            int(df_raw.duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), "tweet_text"].duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), ["tweet_text", "sentiment"]].duplicated().sum()),
        ],
    }
)

blank_text_by_sentiment = (
    df_raw.loc[blank_or_missing_text, "sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="blank_or_missing_rows")
)

display(duplicate_summary)
display(blank_text_by_sentiment)

In [ ]:
# Raw tweet length statistics. These help set later sequence length choices.
df_raw_profile = df_raw.copy()
df_raw_profile["tweet_text_str"] = text_as_string
df_raw_profile["char_len"] = df_raw_profile["tweet_text_str"].str.len()
df_raw_profile["word_len"] = df_raw_profile["tweet_text_str"].str.split().str.len()

length_summary_overall = (
    df_raw_profile[["char_len", "word_len"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

length_summary_by_sentiment = (
    df_raw_profile
    .groupby("sentiment")[["char_len", "word_len"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
length_summary_by_sentiment.columns = [
    f"{feature}_{stat}" for feature, stat in length_summary_by_sentiment.columns.to_flat_index()
]

top_entities_raw = (
    df_raw["entity"]
    .value_counts()
    .rename_axis("entity")
    .reset_index(name="count")
)
top_entities_raw["percent"] = (top_entities_raw["count"] / len(df_raw) * 100).round(2)

display(length_summary_overall)
display(length_summary_by_sentiment)
display(top_entities_raw.head(15))

In [ ]:
# Save Phase 2 audit tables for the report and later phases.
phase2_tables = {
    "phase2_dataset_overview.csv": dataset_overview,
    "phase2_missing_summary.csv": missing_summary,
    "phase2_sentiment_distribution.csv": sentiment_distribution,
    "phase2_sentiment_validation.csv": sentiment_validation,
    "phase2_duplicate_summary.csv": duplicate_summary,
    "phase2_blank_text_by_sentiment.csv": blank_text_by_sentiment,
    "phase2_length_summary_overall.csv": length_summary_overall.reset_index(names="statistic"),
    "phase2_length_summary_by_sentiment.csv": length_summary_by_sentiment.reset_index(),
    "phase2_top_entities_raw.csv": top_entities_raw,
}

for filename, table in phase2_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

phase2_summary_lines = [
    "# Phase 2 Validation Summary",
    "",
    f"- Dataset rows: {len(df_raw):,}",
    f"- Dataset columns: {df_raw.shape[1]:,}",
    f"- Schema matches expected columns: {schema_is_valid}",
    f"- Missing tweet_text values: {int(df_raw['tweet_text'].isna().sum()):,}",
    f"- Blank or missing tweet_text rows: {int(blank_or_missing_text.sum()):,}",
    f"- Exact duplicate rows: {int(df_raw.duplicated().sum()):,}",
    f"- Unique entities: {df_raw['entity'].nunique(dropna=True):,}",
    f"- Unique sentiment labels: {df_raw['sentiment'].nunique(dropna=True):,}",
    f"- Extra raw label outside assignment scope: {', '.join(extra_sentiments) if extra_sentiments else 'None'}",
    "",
    "## Raw Sentiment Distribution",
    "",
]
phase2_summary_lines.extend(
    [
        f"- {row.sentiment}: {int(row.count):,} ({row.percent:.2f}%)"
        for row in sentiment_distribution.itertuples(index=False)
    ]
)
phase2_summary_lines.extend(
    [
        "",
        "## Recommendation For Phase 3",
        "",
        "Create a cleaned working DataFrame by dropping blank or missing tweet text, removing exact duplicate rows, documenting exclusion of Irrelevant, and then applying tweet text cleaning.",
    ]
)
summary_path = REPORTS_DIR / "phase2_validation_summary.md"
summary_path.write_text("\n".join(phase2_summary_lines) + "\n", encoding="utf-8")

print(f"Saved {len(phase2_tables)} Phase 2 tables to {TABLES_DIR}")
print(f"Saved Phase 2 validation summary to {summary_path}")

## Phase 2 Findings To Carry Forward

The raw dataset loads successfully into four columns: `tweet_id`, `entity`, `sentiment`, and `tweet_text`.

Important findings to use in Phase 3:

- The CSV has no header row, so explicit column names are required.
- The raw dataset includes a fourth label, `Irrelevant`, even though the assignment asks for `Negative`, `Neutral`, and `Positive` sentiment classification.
- Blank or missing tweet text and duplicate rows must be handled before modeling.
- Text length statistics should guide sequence padding choices in the RNN phase.

Recommended Phase 3 action: create a cleaned working DataFrame that drops blank or missing tweet text, removes exact duplicate rows, documents the exclusion of `Irrelevant`, and prepares text-cleaning functions for URLs, mentions, hashtags, special characters, tokenization, stop-word removal, and lemmatization or stemming.

## Phase 3: Data Cleaning

Phase 3 creates the cleaned working dataset for downstream EDA, feature engineering, and RNN modeling.

Cleaning decisions:

- Keep the original `twitter_training.csv` unchanged.
- Drop missing or blank tweet text because those rows cannot provide sentiment signal.
- Remove exact duplicate rows to reduce repeated examples and reporting distortion.
- Exclude `Irrelevant` from the main dataset because the assignment rubric targets `Negative`, `Neutral`, and `Positive` sentiment classification.
- Create `model_text` after lowercasing and removing URLs, mentions, hashtags, special characters, and extra whitespace.
- Create `processed_text` after tokenization, stop-word removal, and stemming. Negation words such as `no`, `not`, and `never` are intentionally preserved because they are important for sentiment.
- Drop rows that become empty after text cleaning.

The cleaned dataset is saved under `outputs/data` so later phases can load a stable working file.

In [ ]:
# Phase 3 directories and cleaning resources.
import html
import re

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This compact stop-word list is kept local so the notebook does not depend on downloading
# external NLTK corpora. Negation terms are deliberately excluded from the list.
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him", "his",
    "i", "if", "in", "into", "is", "it", "its", "me", "my", "of", "on",
    "or", "our", "ours", "she", "so", "than", "that", "the", "their", "them",
    "they", "this", "to", "was", "we", "were", "will", "with", "you", "your",
    "yours", "all", "do", "does", "did", "just", "can", "could", "should",
    "would", "about", "above", "after", "again", "against", "am", "any",
    "because", "before", "below", "between", "both", "down", "during", "each",
    "few", "further", "here", "how", "more", "most", "other", "over", "own",
    "same", "some", "such", "then", "there", "these", "those", "through", "too",
    "under", "until", "up", "very", "what", "when", "where", "which", "who",
    "whom", "why", "s", "t", "now", "get", "got", "im", "ll", "re", "ve",
    "u", "ur", "rt"
}
NEGATION_WORDS = {"no", "not", "nor", "never", "none", "cannot", "cant", "dont", "wont"}
STOP_WORDS = STOP_WORDS - NEGATION_WORDS

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
NON_LETTER_RE = re.compile(r"[^a-z\s]")
WHITESPACE_RE = re.compile(r"\s+")

try:
    from nltk.stem import PorterStemmer
    STEMMER = PorterStemmer()
    STEMMER_NAME = "nltk_porter"
except ImportError:
    STEMMER = None
    STEMMER_NAME = "fallback_suffix_stemmer"

print(f"Phase 3 cleaning resources ready. Stemming method: {STEMMER_NAME}")

In [ ]:
# Text cleaning helper functions.
def expand_common_contractions(text):
    """Normalize common English contractions before punctuation removal."""
    replacements = {
        "can't": "can not",
        "cannot": "can not",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
        "'s": " ",
    }
    for pattern, replacement in replacements.items():
        text = text.replace(pattern, replacement)
    return text


def basic_clean_tweet(text):
    """Lowercase and remove Twitter-specific noise and non-letter characters."""
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text).lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = expand_common_contractions(text)
    text = NON_LETTER_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


def tokenize_text(text):
    return [token for token in str(text).split() if token]


def remove_stop_words(tokens):
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 1]


def fallback_stem(word):
    """Small deterministic fallback used only when NLTK is unavailable."""
    if len(word) <= 4:
        return word
    for suffix, replacement in [
        ("ingly", ""),
        ("edly", ""),
        ("ing", ""),
        ("ies", "y"),
        ("ied", "y"),
        ("ed", ""),
        ("ly", ""),
        ("s", ""),
    ]:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: -len(suffix)] + replacement
    return word


def stem_tokens(tokens):
    if STEMMER is not None:
        return [STEMMER.stem(token) for token in tokens]
    return [fallback_stem(token) for token in tokens]


print("Phase 3 helper functions ready.")

In [ ]:
# Build the cleaned working dataset.
df_clean = df_raw.copy()
raw_row_count = len(df_clean)

df_clean["tweet_text_stripped"] = df_clean["tweet_text"].fillna("").astype(str).str.strip()
blank_or_missing_mask = df_clean["tweet_text_stripped"].eq("")
blank_or_missing_removed = int(blank_or_missing_mask.sum())
df_clean = df_clean.loc[~blank_or_missing_mask].copy()

before_duplicate_drop = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=RAW_COLUMNS).copy()
duplicate_rows_removed = before_duplicate_drop - len(df_clean)

before_label_filter = len(df_clean)
df_clean = df_clean.loc[df_clean["sentiment"].isin(sorted(ASSIGNMENT_SENTIMENTS))].copy()
irrelevant_rows_removed = before_label_filter - len(df_clean)

df_clean["model_text"] = df_clean["tweet_text"].apply(basic_clean_tweet)
empty_after_cleaning_mask = df_clean["model_text"].str.strip().eq("")
empty_after_cleaning_removed = int(empty_after_cleaning_mask.sum())
df_clean = df_clean.loc[~empty_after_cleaning_mask].copy()

df_clean["tokens"] = df_clean["model_text"].apply(tokenize_text)
df_clean["tokens_no_stopwords"] = df_clean["tokens"].apply(remove_stop_words)
df_clean["stemmed_tokens"] = df_clean["tokens_no_stopwords"].apply(stem_tokens)
df_clean["processed_text"] = df_clean["stemmed_tokens"].apply(" ".join)

df_clean["raw_char_len"] = df_clean["tweet_text"].astype(str).str.len()
df_clean["raw_word_len"] = df_clean["tweet_text"].astype(str).str.split().str.len()
df_clean["model_word_len"] = df_clean["tokens"].apply(len)
df_clean["processed_word_len"] = df_clean["stemmed_tokens"].apply(len)

# Convert token lists to space-separated strings for inspection, then export only
# stable downstream columns to keep the cleaned artifact compact.
df_clean_output = df_clean.copy()
df_clean_output["tokens"] = df_clean_output["tokens"].apply(" ".join)
df_clean_output["tokens_no_stopwords"] = df_clean_output["tokens_no_stopwords"].apply(" ".join)
df_clean_output["stemmed_tokens"] = df_clean_output["stemmed_tokens"].apply(" ".join)
cleaned_export_columns = [
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "raw_char_len",
    "raw_word_len",
    "model_word_len",
    "processed_word_len",
]
df_clean_export = df_clean_output[cleaned_export_columns].copy()

cleaned_dataset_path = DATA_OUTPUT_DIR / "twitter_training_cleaned_phase3.csv"
df_clean_export.to_csv(cleaned_dataset_path, index=False)

print(f"Raw rows: {raw_row_count:,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Cleaned dataset saved to: {cleaned_dataset_path}")
display(df_clean_export[["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "processed_text"]].head())

In [ ]:
# Phase 3 cleaning audit tables.
cleaning_steps = pd.DataFrame(
    {
        "step": [
            "raw_rows",
            "drop_blank_or_missing_tweet_text",
            "drop_exact_duplicate_rows",
            "exclude_irrelevant_label",
            "drop_empty_text_after_cleaning",
            "final_cleaned_rows",
        ],
        "rows_removed": [
            0,
            blank_or_missing_removed,
            duplicate_rows_removed,
            irrelevant_rows_removed,
            empty_after_cleaning_removed,
            0,
        ],
        "rows_remaining": [
            raw_row_count,
            raw_row_count - blank_or_missing_removed,
            raw_row_count - blank_or_missing_removed - duplicate_rows_removed,
            raw_row_count - blank_or_missing_removed - duplicate_rows_removed - irrelevant_rows_removed,
            len(df_clean),
            len(df_clean),
        ],
    }
)

phase3_sentiment_distribution = (
    df_clean["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)
phase3_sentiment_distribution["percent"] = (
    phase3_sentiment_distribution["count"] / len(df_clean) * 100
).round(2)

phase3_length_summary = (
    df_clean[["raw_word_len", "model_word_len", "processed_word_len"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
    .reset_index(names="statistic")
)

phase3_text_quality_summary = pd.DataFrame(
    {
        "metric": [
            "rows_with_urls_in_raw_text",
            "rows_with_mentions_in_raw_text",
            "rows_with_hashtags_in_raw_text",
            "rows_with_empty_model_text_after_cleaning_removed",
            "rows_with_empty_processed_text_remaining",
            "stemming_method",
        ],
        "value": [
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(URL_RE).sum()),
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(MENTION_RE).sum()),
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(HASHTAG_RE).sum()),
            empty_after_cleaning_removed,
            int(df_clean["processed_text"].str.strip().eq("").sum()),
            STEMMER_NAME,
        ],
    }
)

phase3_sample_cleaned_rows = df_clean_export[
    ["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "processed_text"]
].sample(12, random_state=42)

phase3_tables = {
    "phase3_cleaning_steps.csv": cleaning_steps,
    "phase3_cleaned_sentiment_distribution.csv": phase3_sentiment_distribution,
    "phase3_length_summary.csv": phase3_length_summary,
    "phase3_text_quality_summary.csv": phase3_text_quality_summary,
    "phase3_sample_cleaned_rows.csv": phase3_sample_cleaned_rows,
}

for filename, table in phase3_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

display(cleaning_steps)
display(phase3_sentiment_distribution)
display(phase3_text_quality_summary)

In [ ]:
# Save a concise Phase 3 report summary.
phase3_summary_lines = [
    "# Phase 3 Data Cleaning Summary",
    "",
    f"- Raw rows loaded: {raw_row_count:,}",
    f"- Removed blank or missing tweet text rows: {blank_or_missing_removed:,}",
    f"- Removed exact duplicate rows after blank-text removal: {duplicate_rows_removed:,}",
    f"- Removed rows with Irrelevant label: {irrelevant_rows_removed:,}",
    f"- Removed rows that became empty after text cleaning: {empty_after_cleaning_removed:,}",
    f"- Final cleaned rows: {len(df_clean):,}",
    f"- Stemming method used in this run: {STEMMER_NAME}",
    f"- Cleaned dataset: {cleaned_dataset_path}",
    "",
    "## Final Cleaned Sentiment Distribution",
    "",
]
phase3_summary_lines.extend(
    [
        f"- {row.sentiment}: {int(row.count):,} ({row.percent:.2f}%)"
        for row in phase3_sentiment_distribution.itertuples(index=False)
    ]
)
phase3_summary_lines.extend(
    [
        "",
        "## Cleaning Notes",
        "",
        "The original raw CSV was not modified. The cleaned dataset excludes Irrelevant because the graded assignment defines a three-class sentiment task. The notebook keeps both model_text and processed_text so later RNN work can use a sequence-friendly text representation while EDA and rubric documentation can show stop-word removal and stemming.",
    ]
)

phase3_summary_path = REPORTS_DIR / "phase3_cleaning_summary.md"
phase3_summary_path.write_text("\n".join(phase3_summary_lines) + "\n", encoding="utf-8")

print(f"Saved Phase 3 tables to {TABLES_DIR}")
print(f"Saved Phase 3 summary to {phase3_summary_path}")

## Phase 3 Findings To Carry Forward

The cleaned dataset is now ready for EDA and feature engineering.

Carry-forward points:

- `model_text` should be used for RNN sequence modeling because it preserves more word order and context.
- `processed_text` and `stemmed_tokens` are useful for rubric-visible preprocessing, top-word analysis, and word clouds.
- The final model should remain a three-class classifier: `Negative`, `Neutral`, and `Positive`.
- Later train/test splits should be created from the cleaned dataset, not the raw CSV.
- Sequence length decisions should use the cleaned `model_word_len` distribution rather than raw tweet length alone.

## Phase 4: Text Preprocessing

Phase 4 converts the cleaned text fields into stable text-processing artifacts for EDA and later feature engineering.

Scope of this phase:

- Load the Phase 3 cleaned dataset.
- Validate that only `Negative`, `Neutral`, and `Positive` labels remain.
- Create `model_tokens` from `model_text` for later RNN sequence construction.
- Create `analysis_text` and `analysis_tokens` for EDA. `analysis_text` uses `processed_text` where available, and falls back to `model_text` when stop-word removal and stemming leave a row empty.
- Save token-count summaries, top token frequency tables, label mapping metadata, and a compact preprocessed dataset.

This phase does not create padded integer sequences, TF-IDF matrices, or embeddings. Those belong to the Feature Engineering phase.

In [ ]:
# Phase 4: load the cleaned dataset created in Phase 3.
from collections import Counter

PHASE3_CLEANED_PATH = DATA_OUTPUT_DIR / "twitter_training_cleaned_phase3.csv"
PHASE4_PREPROCESSED_PATH = DATA_OUTPUT_DIR / "twitter_text_preprocessed_phase4.csv"

df_text = pd.read_csv(PHASE3_CLEANED_PATH)

required_phase3_columns = {
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "raw_word_len",
    "model_word_len",
    "processed_word_len",
}
missing_phase3_columns = sorted(required_phase3_columns - set(df_text.columns))
if missing_phase3_columns:
    raise ValueError(f"Missing expected Phase 3 columns: {missing_phase3_columns}")

remaining_labels = set(df_text["sentiment"].dropna().unique())
if remaining_labels != ASSIGNMENT_SENTIMENTS:
    raise ValueError(f"Unexpected cleaned sentiment labels: {sorted(remaining_labels)}")

print(f"Loaded Phase 3 cleaned dataset: {PHASE3_CLEANED_PATH}")
print(f"Shape: {df_text.shape[0]:,} rows x {df_text.shape[1]:,} columns")
print(f"Labels: {sorted(remaining_labels)}")

In [ ]:
# Create reusable text and token fields for later phases.
df_preprocessed = df_text.copy()

df_preprocessed["model_text"] = df_preprocessed["model_text"].fillna("").astype(str).str.strip()
df_preprocessed["processed_text"] = df_preprocessed["processed_text"].fillna("").astype(str).str.strip()

empty_processed_text_before_fallback = int(df_preprocessed["processed_text"].eq("").sum())
df_preprocessed["analysis_text"] = np.where(
    df_preprocessed["processed_text"].eq(""),
    df_preprocessed["model_text"],
    df_preprocessed["processed_text"],
)

df_preprocessed["model_tokens"] = df_preprocessed["model_text"].str.split()
df_preprocessed["analysis_tokens"] = df_preprocessed["analysis_text"].str.split()
df_preprocessed["model_token_count"] = df_preprocessed["model_tokens"].apply(len)
df_preprocessed["analysis_token_count"] = df_preprocessed["analysis_tokens"].apply(len)

empty_model_tokens = int(df_preprocessed["model_token_count"].eq(0).sum())
empty_analysis_tokens = int(df_preprocessed["analysis_token_count"].eq(0).sum())
if empty_model_tokens or empty_analysis_tokens:
    raise ValueError(
        f"Unexpected empty token rows: model={empty_model_tokens}, analysis={empty_analysis_tokens}"
    )

# Store tokenized text as space-separated strings for readable CSV artifacts.
df_preprocessed_export = df_preprocessed.copy()
df_preprocessed_export["model_tokens"] = df_preprocessed_export["model_tokens"].apply(" ".join)
df_preprocessed_export["analysis_tokens"] = df_preprocessed_export["analysis_tokens"].apply(" ".join)

phase4_export_columns = [
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "analysis_text",
    "raw_word_len",
    "model_token_count",
    "analysis_token_count",
]
df_preprocessed_export = df_preprocessed_export[phase4_export_columns].copy()
df_preprocessed_export.to_csv(PHASE4_PREPROCESSED_PATH, index=False)

display(df_preprocessed[["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "analysis_text", "model_token_count", "analysis_token_count"]].head())
print(f"Preprocessed dataset saved to: {PHASE4_PREPROCESSED_PATH}")

In [ ]:
# Phase 4 summaries and reusable metadata.
label_order = ["Negative", "Neutral", "Positive"]
label_mapping = pd.DataFrame(
    {
        "sentiment": label_order,
        "label_id": list(range(len(label_order))),
        "note": ["negative class", "neutral class", "positive class"],
    }
)

model_p99 = float(df_preprocessed["model_token_count"].quantile(0.99))
recommended_max_sequence_length = int(np.ceil(model_p99 / 10) * 10)

phase4_overview = pd.DataFrame(
    {
        "metric": [
            "rows_preprocessed",
            "unique_entities",
            "unique_sentiments",
            "empty_processed_text_before_analysis_fallback",
            "empty_model_token_rows",
            "empty_analysis_token_rows",
            "model_token_count_p95",
            "model_token_count_p99",
            "recommended_max_sequence_length_for_later_padding",
        ],
        "value": [
            f"{len(df_preprocessed):,}",
            f"{df_preprocessed['entity'].nunique():,}",
            f"{df_preprocessed['sentiment'].nunique():,}",
            f"{empty_processed_text_before_fallback:,}",
            f"{empty_model_tokens:,}",
            f"{empty_analysis_tokens:,}",
            f"{float(df_preprocessed['model_token_count'].quantile(0.95)):.2f}",
            f"{model_p99:.2f}",
            f"{recommended_max_sequence_length:,}",
        ],
    }
)

phase4_token_length_summary = (
    df_preprocessed[["model_token_count", "analysis_token_count"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
    .reset_index(names="statistic")
)

phase4_token_length_by_sentiment = (
    df_preprocessed
    .groupby("sentiment")[["model_token_count", "analysis_token_count"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
phase4_token_length_by_sentiment.columns = [
    f"{feature}_{stat}" for feature, stat in phase4_token_length_by_sentiment.columns.to_flat_index()
]
phase4_token_length_by_sentiment = phase4_token_length_by_sentiment.reset_index()

model_text_sentiment_duplicates = int(df_preprocessed.duplicated(["model_text", "sentiment"]).sum())
model_text_duplicates = int(df_preprocessed.duplicated(["model_text"]).sum())
analysis_text_sentiment_duplicates = int(df_preprocessed.duplicated(["analysis_text", "sentiment"]).sum())
model_text_label_counts = df_preprocessed.groupby("model_text")["sentiment"].nunique()
conflicting_model_texts = model_text_label_counts[model_text_label_counts > 1].index
conflicting_model_text_count = len(conflicting_model_texts)
rows_in_conflicting_model_text_groups = int(
    df_preprocessed["model_text"].isin(conflicting_model_texts).sum()
)

phase4_duplicate_text_audit = pd.DataFrame(
    {
        "metric": [
            "duplicate_model_text_rows",
            "duplicate_model_text_and_sentiment_rows",
            "duplicate_analysis_text_and_sentiment_rows",
            "model_text_values_with_multiple_sentiment_labels",
            "rows_in_conflicting_model_text_groups",
        ],
        "value": [
            model_text_duplicates,
            model_text_sentiment_duplicates,
            analysis_text_sentiment_duplicates,
            conflicting_model_text_count,
            rows_in_conflicting_model_text_groups,
        ],
        "phase5_or_phase6_action": [
            "Audit before EDA; repeated cleaned text can dominate frequency counts.",
            "Consider deduplicating text-label pairs inside the modeling dataset if leakage risk is high.",
            "Use with care for EDA only; analysis_text is more aggressively reduced than model_text.",
            "Inspect before modeling because the same cleaned text carries multiple labels.",
            "Avoid random leakage by handling or grouping these cases before final train/test split.",
        ],
    }
)

phase4_conflicting_model_text_examples = (
    df_preprocessed.loc[
        df_preprocessed["model_text"].isin(conflicting_model_texts),
        ["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "analysis_text"],
    ]
    .sort_values(["model_text", "sentiment", "tweet_id"])
    .head(60)
)

display(phase4_overview)
display(label_mapping)
display(phase4_token_length_summary)
display(phase4_duplicate_text_audit)

In [ ]:
# Token frequency tables for EDA readiness. These are descriptive, not model vocabularies.
def token_frequency_table(token_series, top_n=100):
    counter = Counter()
    for tokens in token_series:
        counter.update(tokens)
    total_tokens = sum(counter.values())
    rows = [
        {
            "token": token,
            "count": count,
            "percent_of_tokens": round(count / total_tokens * 100, 4) if total_tokens else 0,
        }
        for token, count in counter.most_common(top_n)
    ]
    return pd.DataFrame(rows)


phase4_top_analysis_tokens = token_frequency_table(df_preprocessed["analysis_tokens"], top_n=100)
phase4_top_model_tokens = token_frequency_table(df_preprocessed["model_tokens"], top_n=100)

by_sentiment_frames = []
for sentiment, group in df_preprocessed.groupby("sentiment"):
    table = token_frequency_table(group["analysis_tokens"], top_n=50)
    table.insert(0, "sentiment", sentiment)
    table.insert(1, "rank", range(1, len(table) + 1))
    by_sentiment_frames.append(table)

phase4_top_analysis_tokens_by_sentiment = pd.concat(by_sentiment_frames, ignore_index=True)

display(phase4_top_analysis_tokens.head(20))
display(phase4_top_analysis_tokens_by_sentiment.head(15))

In [ ]:
# Save Phase 4 artifacts for EDA and later feature engineering.
phase4_sample_rows = df_preprocessed_export[
    ["tweet_id", "entity", "sentiment", "model_text", "analysis_text", "model_token_count", "analysis_token_count"]
].sample(12, random_state=42)

phase4_tables = {
    "phase4_preprocessing_overview.csv": phase4_overview,
    "phase4_label_mapping.csv": label_mapping,
    "phase4_token_length_summary.csv": phase4_token_length_summary,
    "phase4_token_length_by_sentiment.csv": phase4_token_length_by_sentiment,
    "phase4_top_analysis_tokens.csv": phase4_top_analysis_tokens,
    "phase4_top_model_tokens.csv": phase4_top_model_tokens,
    "phase4_top_analysis_tokens_by_sentiment.csv": phase4_top_analysis_tokens_by_sentiment,
    "phase4_duplicate_text_audit.csv": phase4_duplicate_text_audit,
    "phase4_conflicting_model_text_examples.csv": phase4_conflicting_model_text_examples,
    "phase4_sample_preprocessed_rows.csv": phase4_sample_rows,
}

for filename, table in phase4_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

phase4_summary_lines = [
    "# Phase 4 Text Preprocessing Summary",
    "",
    f"- Source cleaned dataset: {PHASE3_CLEANED_PATH}",
    f"- Preprocessed dataset: {PHASE4_PREPROCESSED_PATH}",
    f"- Rows preprocessed: {len(df_preprocessed):,}",
    f"- Empty processed_text rows handled with model_text fallback: {empty_processed_text_before_fallback:,}",
    f"- Empty model token rows after preprocessing: {empty_model_tokens:,}",
    f"- Empty analysis token rows after preprocessing: {empty_analysis_tokens:,}",
    f"- Model token count p95: {float(df_preprocessed['model_token_count'].quantile(0.95)):.2f}",
    f"- Model token count p99: {model_p99:.2f}",
    f"- Recommended max sequence length for later padding: {recommended_max_sequence_length}",
    f"- Duplicate model_text and sentiment rows to audit before modeling: {model_text_sentiment_duplicates:,}",
    f"- Cleaned model_text values with multiple sentiment labels: {conflicting_model_text_count:,}",
    f"- Rows in conflicting model_text groups: {rows_in_conflicting_model_text_groups:,}",
    "",
    "## Label Mapping For Later Modeling",
    "",
]
phase4_summary_lines.extend(
    [f"- {row.sentiment}: {row.label_id}" for row in label_mapping.itertuples(index=False)]
)
phase4_summary_lines.extend(
    [
        "",
        "## Notes",
        "",
        "The Phase 4 token frequency tables are descriptive preprocessing artifacts for EDA. They are not the final model vocabulary. The model vocabulary must be built from the training split only during feature engineering to avoid data leakage.",
        "",
        "The duplicate-text audit is intentionally not applied as a row-removal step in this phase. It should guide Phase 6 train/test splitting and optional text-label deduplication so repeated cleaned text does not leak across splits.",
    ]
)

phase4_summary_path = REPORTS_DIR / "phase4_text_preprocessing_summary.md"
phase4_summary_path.write_text("\n".join(phase4_summary_lines) + "\n", encoding="utf-8")

print(f"Saved preprocessed dataset to {PHASE4_PREPROCESSED_PATH}")
print(f"Saved {len(phase4_tables)} Phase 4 tables to {TABLES_DIR}")
print(f"Saved Phase 4 summary to {phase4_summary_path}")

## Phase 4 Findings To Carry Forward

Phase 4 produces a stable preprocessed text dataset and descriptive token artifacts.

Carry-forward points:

- Use `model_text` or `model_tokens` when creating RNN token sequences in Feature Engineering.
- Use `analysis_text` and `analysis_tokens` for Phase 5 EDA word frequencies and word clouds.
- Use the saved label mapping consistently when converting sentiment labels for modeling.
- The recommended maximum sequence length is based on the cleaned `model_token_count` distribution and should be revisited only if later train/test split statistics differ materially.
- Do not treat Phase 4 frequency tables as the model vocabulary; the model vocabulary must be learned from the training split only.

# Part 2: Exploratory Data Analysis

## Phase 5: Exploratory Data Analysis

Phase 5 explores the cleaned and preprocessed three-class dataset before feature engineering and modeling.

EDA deliverables for the rubric:

- Basic statistics for the cleaned dataset.
- Sentiment distribution.
- Entity/topic distribution.
- Top words by sentiment.
- Word-cloud-style visualizations for positive and negative tweets.
- Tweet length patterns by sentiment.
- Written insights and modeling implications.

The figures in this phase are saved as SVG files under `outputs/figures`.

In [ ]:
# Phase 5: load the preprocessed dataset from Phase 4.
import math
import html as html_lib

PHASE4_PREPROCESSED_PATH = DATA_OUTPUT_DIR / "twitter_text_preprocessed_phase4.csv"
df_eda = pd.read_csv(PHASE4_PREPROCESSED_PATH)

required_phase5_columns = {
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "analysis_text",
    "raw_word_len",
    "model_token_count",
    "analysis_token_count",
}
missing_phase5_columns = sorted(required_phase5_columns - set(df_eda.columns))
if missing_phase5_columns:
    raise ValueError(f"Missing expected Phase 5 columns: {missing_phase5_columns}")

if set(df_eda["sentiment"].unique()) != ASSIGNMENT_SENTIMENTS:
    raise ValueError("Phase 5 expected only Negative, Neutral, and Positive labels.")

df_eda["analysis_tokens"] = df_eda["analysis_text"].fillna("").astype(str).str.split()
df_eda["model_tokens"] = df_eda["model_text"].fillna("").astype(str).str.split()

print(f"Loaded Phase 4 preprocessed dataset: {PHASE4_PREPROCESSED_PATH}")
print(f"EDA shape: {df_eda.shape[0]:,} rows x {df_eda.shape[1]:,} columns")
print(f"Sentiment labels: {sorted(df_eda['sentiment'].unique())}")

In [ ]:
# Phase 5 summary statistics and distributions.
sentiment_distribution_eda = (
    df_eda["sentiment"]
    .value_counts()
    .reindex(["Negative", "Positive", "Neutral"])
    .rename_axis("sentiment")
    .reset_index(name="count")
)
sentiment_distribution_eda["percent"] = (
    sentiment_distribution_eda["count"] / len(df_eda) * 100
).round(2)

entity_distribution_eda = (
    df_eda["entity"]
    .value_counts()
    .rename_axis("entity")
    .reset_index(name="count")
)
entity_distribution_eda["percent"] = (entity_distribution_eda["count"] / len(df_eda) * 100).round(2)

phase5_basic_statistics = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_tweet_ids",
            "unique_entities",
            "unique_sentiments",
            "mode_sentiment",
            "mode_entity",
            "mean_raw_word_len",
            "median_raw_word_len",
            "mean_model_token_count",
            "median_model_token_count",
            "p95_model_token_count",
            "p99_model_token_count",
        ],
        "value": [
            f"{len(df_eda):,}",
            f"{df_eda.shape[1]:,}",
            f"{df_eda['tweet_id'].nunique():,}",
            f"{df_eda['entity'].nunique():,}",
            f"{df_eda['sentiment'].nunique():,}",
            df_eda["sentiment"].mode().iat[0],
            df_eda["entity"].mode().iat[0],
            f"{df_eda['raw_word_len'].mean():.2f}",
            f"{df_eda['raw_word_len'].median():.2f}",
            f"{df_eda['model_token_count'].mean():.2f}",
            f"{df_eda['model_token_count'].median():.2f}",
            f"{df_eda['model_token_count'].quantile(0.95):.2f}",
            f"{df_eda['model_token_count'].quantile(0.99):.2f}",
        ],
    }
)

length_by_sentiment_eda = (
    df_eda
    .groupby("sentiment")
    .agg(
        rows=("model_token_count", "count"),
        raw_word_mean=("raw_word_len", "mean"),
        raw_word_median=("raw_word_len", "median"),
        model_token_mean=("model_token_count", "mean"),
        model_token_median=("model_token_count", "median"),
        model_token_p25=("model_token_count", lambda values: values.quantile(0.25)),
        model_token_p75=("model_token_count", lambda values: values.quantile(0.75)),
        model_token_p90=("model_token_count", lambda values: values.quantile(0.90)),
        model_token_p95=("model_token_count", lambda values: values.quantile(0.95)),
        analysis_token_mean=("analysis_token_count", "mean"),
        analysis_token_median=("analysis_token_count", "median"),
    )
    .round(2)
    .reset_index()
)

display(phase5_basic_statistics)
display(sentiment_distribution_eda)
display(entity_distribution_eda.head(15))
display(length_by_sentiment_eda)

In [ ]:
# Phase 5 token frequencies by sentiment.
def phase5_token_frequency(token_lists, top_n=50):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    total = sum(counter.values())
    return pd.DataFrame(
        [
            {
                "token": token,
                "count": count,
                "percent_of_tokens": round(count / total * 100, 4) if total else 0,
            }
            for token, count in counter.most_common(top_n)
        ]
    )


top_tokens_by_sentiment_frames = []
top_tokens_lookup = {}
for sentiment in ["Negative", "Neutral", "Positive"]:
    sentiment_tokens = df_eda.loc[df_eda["sentiment"].eq(sentiment), "analysis_tokens"]
    token_table = phase5_token_frequency(sentiment_tokens, top_n=60)
    token_table.insert(0, "sentiment", sentiment)
    token_table.insert(1, "rank", range(1, len(token_table) + 1))
    top_tokens_by_sentiment_frames.append(token_table)
    top_tokens_lookup[sentiment] = token_table.copy()

phase5_top_tokens_by_sentiment = pd.concat(top_tokens_by_sentiment_frames, ignore_index=True)

display(phase5_top_tokens_by_sentiment.groupby("sentiment").head(12))

In [ ]:
# SVG visualization helpers. These avoid runtime dependency on plotting libraries.
def svg_text(text):
    return html_lib.escape(str(text), quote=True)


def save_svg(content, path, width, height):
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" role="img">\n{content}\n</svg>\n'
    )
    path.write_text(svg, encoding="utf-8")


def save_vertical_bar_chart(data, label_col, value_col, title, y_label, path, color="#2f6f9f", width=920, height=560):
    margin_left, margin_right, margin_top, margin_bottom = 90, 40, 80, 130
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    max_value = max(float(v) for v in data[value_col])
    bar_gap = 24
    bar_width = (plot_width - bar_gap * (len(data) - 1)) / len(data)
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="35" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">{svg_text(title)}</text>',
        f'<line x1="{margin_left}" y1="{height-margin_bottom}" x2="{width-margin_right}" y2="{height-margin_bottom}" stroke="#333" stroke-width="1"/>',
        f'<line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{height-margin_bottom}" stroke="#333" stroke-width="1"/>',
        f'<text x="25" y="{margin_top + plot_height/2}" text-anchor="middle" transform="rotate(-90 25 {margin_top + plot_height/2})" font-family="Arial" font-size="14" fill="#333">{svg_text(y_label)}</text>',
    ]
    for i, row in data.reset_index(drop=True).iterrows():
        value = float(row[value_col])
        bar_height = 0 if max_value == 0 else value / max_value * plot_height
        x = margin_left + i * (bar_width + bar_gap)
        y = height - margin_bottom - bar_height
        label = row[label_col]
        percent = row.get("percent", None)
        value_label = f"{int(value):,}"
        if percent is not None:
            value_label += f" ({float(percent):.1f}%)"
        parts.extend([
            f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" rx="4" fill="{color}"/>',
            f'<text x="{x + bar_width/2:.1f}" y="{y - 10:.1f}" text-anchor="middle" font-family="Arial" font-size="13" fill="#222">{svg_text(value_label)}</text>',
            f'<text x="{x + bar_width/2:.1f}" y="{height - margin_bottom + 28}" text-anchor="middle" font-family="Arial" font-size="14" fill="#222">{svg_text(label)}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_horizontal_bar_chart(data, label_col, value_col, title, path, color="#4b8f69", width=980, height=640):
    margin_left, margin_right, margin_top, margin_bottom = 210, 80, 75, 45
    plot_width = width - margin_left - margin_right
    row_height = (height - margin_top - margin_bottom) / len(data)
    max_value = max(float(v) for v in data[value_col])
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">{svg_text(title)}</text>',
    ]
    for i, row in data.reset_index(drop=True).iterrows():
        value = float(row[value_col])
        bar_width = 0 if max_value == 0 else value / max_value * plot_width
        y = margin_top + i * row_height
        parts.extend([
            f'<text x="{margin_left - 12}" y="{y + row_height*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#222">{svg_text(row[label_col])}</text>',
            f'<rect x="{margin_left}" y="{y + row_height*0.18:.1f}" width="{bar_width:.1f}" height="{row_height*0.58:.1f}" rx="3" fill="{color}"/>',
            f'<text x="{margin_left + bar_width + 8:.1f}" y="{y + row_height*0.62:.1f}" font-family="Arial" font-size="12" fill="#222">{int(value):,}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_length_box_svg(length_table, path, width=920, height=460):
    colors = {"Negative": "#b4494b", "Neutral": "#607d8b", "Positive": "#3f8f61"}
    axis_max = max(60, int(math.ceil(length_table["model_token_p95"].max() / 10) * 10))
    margin_left, margin_right, margin_top, margin_bottom = 150, 70, 80, 70
    plot_width = width - margin_left - margin_right
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Tweet Length By Sentiment</text>',
        f'<text x="{width/2}" y="{height-20}" text-anchor="middle" font-family="Arial" font-size="14" fill="#333">Model token count</text>',
    ]
    for tick in range(0, axis_max + 1, 10):
        x = margin_left + tick / axis_max * plot_width
        parts.append(f'<line x1="{x:.1f}" y1="{margin_top-8}" x2="{x:.1f}" y2="{height-margin_bottom+8}" stroke="#e0e0e0"/>')
        parts.append(f'<text x="{x:.1f}" y="{height-margin_bottom+28}" text-anchor="middle" font-family="Arial" font-size="12" fill="#555">{tick}</text>')
    for i, row in length_table.reset_index(drop=True).iterrows():
        y = margin_top + 70 * i
        sentiment = row["sentiment"]
        color = colors.get(sentiment, "#555")
        x25 = margin_left + row["model_token_p25"] / axis_max * plot_width
        x50 = margin_left + row["model_token_median"] / axis_max * plot_width
        x75 = margin_left + row["model_token_p75"] / axis_max * plot_width
        x90 = margin_left + row["model_token_p90"] / axis_max * plot_width
        x95 = margin_left + row["model_token_p95"] / axis_max * plot_width
        parts.extend([
            f'<text x="{margin_left-20}" y="{y+7}" text-anchor="end" font-family="Arial" font-size="15" font-weight="700" fill="#222">{svg_text(sentiment)}</text>',
            f'<line x1="{x25:.1f}" y1="{y}" x2="{x95:.1f}" y2="{y}" stroke="{color}" stroke-width="3"/>',
            f'<rect x="{x25:.1f}" y="{y-15}" width="{max(x75-x25, 1):.1f}" height="30" fill="{color}" opacity="0.35" stroke="{color}"/>',
            f'<line x1="{x50:.1f}" y1="{y-18}" x2="{x50:.1f}" y2="{y+18}" stroke="{color}" stroke-width="4"/>',
            f'<line x1="{x90:.1f}" y1="{y-13}" x2="{x90:.1f}" y2="{y+13}" stroke="{color}" stroke-width="2" stroke-dasharray="4 3"/>',
            f'<line x1="{x95:.1f}" y1="{y-13}" x2="{x95:.1f}" y2="{y+13}" stroke="{color}" stroke-width="2"/>',
            f'<text x="{x50:.1f}" y="{y+38}" text-anchor="middle" font-family="Arial" font-size="11" fill="#333">median {row["model_token_median"]:.0f}</text>',
            f'<text x="{x95:.1f}" y="{y-24}" text-anchor="middle" font-family="Arial" font-size="11" fill="#333">p95 {row["model_token_p95"]:.0f}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_word_cloud_svg(token_table, title, path, color_palette, width=1000, height=620):
    top_words = token_table.head(70).copy()
    if top_words.empty:
        save_svg("", path, width, height)
        return
    min_count = top_words["count"].min()
    max_count = top_words["count"].max()
    x, y = 45, 92
    line_height = 0
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="38" text-anchor="middle" font-family="Arial" font-size="25" font-weight="700" fill="#222">{svg_text(title)}</text>',
    ]
    for i, row in top_words.reset_index(drop=True).iterrows():
        if max_count == min_count:
            font_size = 28
        else:
            scaled = (math.sqrt(row["count"]) - math.sqrt(min_count)) / (math.sqrt(max_count) - math.sqrt(min_count))
            font_size = 15 + scaled * 48
        word = str(row["token"])
        estimated_width = len(word) * font_size * 0.58
        if x + estimated_width > width - 45:
            x = 45
            y += line_height + 16
            line_height = 0
        if y > height - 35:
            break
        color = color_palette[i % len(color_palette)]
        parts.append(
            f'<text x="{x:.1f}" y="{y:.1f}" font-family="Arial" font-size="{font_size:.1f}" font-weight="700" fill="{color}">{svg_text(word)}</text>'
        )
        x += estimated_width + 24
        line_height = max(line_height, font_size)
    save_svg("\n".join(parts), path, width, height)


def display_svg_file(path):
    try:
        from IPython.display import SVG
        display(SVG(filename=str(path)))
    except Exception:
        print(path)


print("Phase 5 SVG helper functions ready.")

In [ ]:
# Generate Phase 5 figures.
phase5_figure_paths = {
    "sentiment_distribution": FIGURES_DIR / "phase5_sentiment_distribution.svg",
    "top_entities": FIGURES_DIR / "phase5_top_entities.svg",
    "length_by_sentiment": FIGURES_DIR / "phase5_tweet_length_by_sentiment.svg",
    "negative_word_cloud": FIGURES_DIR / "phase5_wordcloud_negative.svg",
    "positive_word_cloud": FIGURES_DIR / "phase5_wordcloud_positive.svg",
    "top_tokens_by_sentiment": FIGURES_DIR / "phase5_top_tokens_by_sentiment.svg",
}

save_vertical_bar_chart(
    sentiment_distribution_eda,
    label_col="sentiment",
    value_col="count",
    title="Sentiment Distribution",
    y_label="Tweet count",
    path=phase5_figure_paths["sentiment_distribution"],
    color="#2f6f9f",
)

save_horizontal_bar_chart(
    entity_distribution_eda.head(15),
    label_col="entity",
    value_col="count",
    title="Top 15 Entities In Cleaned Dataset",
    path=phase5_figure_paths["top_entities"],
    color="#5a6f8f",
)

save_length_box_svg(length_by_sentiment_eda, phase5_figure_paths["length_by_sentiment"])

save_word_cloud_svg(
    top_tokens_lookup["Negative"],
    "Negative Tweet Word Cloud",
    phase5_figure_paths["negative_word_cloud"],
    color_palette=["#7f1d1d", "#b4494b", "#6b2d2d", "#993f3f", "#333333"],
)

save_word_cloud_svg(
    top_tokens_lookup["Positive"],
    "Positive Tweet Word Cloud",
    phase5_figure_paths["positive_word_cloud"],
    color_palette=["#1f6f43", "#3f8f61", "#2f6f9f", "#4f7f50", "#333333"],
)

# Top-token chart uses the top 10 per sentiment for readability.
top_10_token_panels = phase5_top_tokens_by_sentiment.groupby("sentiment").head(10).copy()
panel_parts = ['<rect width="100%" height="100%" fill="#ffffff"/>']
panel_width, panel_height = 1080, 900
panel_parts.append('<text x="540" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Top Analysis Tokens By Sentiment</text>')
panel_colors = {"Negative": "#b4494b", "Neutral": "#607d8b", "Positive": "#3f8f61"}
for panel_index, sentiment in enumerate(["Negative", "Neutral", "Positive"]):
    panel = top_10_token_panels[top_10_token_panels["sentiment"].eq(sentiment)].reset_index(drop=True)
    y_base = 70 + panel_index * 265
    max_count = panel["count"].max()
    panel_parts.append(f'<text x="40" y="{y_base}" font-family="Arial" font-size="20" font-weight="700" fill="#222">{sentiment}</text>')
    for i, row in panel.iterrows():
        y = y_base + 28 + i * 22
        bar_width = row["count"] / max_count * 600
        panel_parts.append(f'<text x="210" y="{y+13}" text-anchor="end" font-family="Arial" font-size="12" fill="#222">{svg_text(row["token"])}</text>')
        panel_parts.append(f'<rect x="225" y="{y}" width="{bar_width:.1f}" height="16" rx="3" fill="{panel_colors[sentiment]}"/>')
        panel_parts.append(f'<text x="{230 + bar_width:.1f}" y="{y+13}" font-family="Arial" font-size="12" fill="#222">{int(row["count"]):,}</text>')
save_svg("\n".join(panel_parts), phase5_figure_paths["top_tokens_by_sentiment"], panel_width, panel_height)

for figure_name, figure_path in phase5_figure_paths.items():
    print(f"{figure_name}: {figure_path}")

In [ ]:
# Save Phase 5 tables and written insights.
phase5_tables = {
    "phase5_basic_statistics.csv": phase5_basic_statistics,
    "phase5_sentiment_distribution.csv": sentiment_distribution_eda,
    "phase5_entity_distribution.csv": entity_distribution_eda,
    "phase5_length_by_sentiment.csv": length_by_sentiment_eda,
    "phase5_top_tokens_by_sentiment.csv": phase5_top_tokens_by_sentiment,
}

for filename, table in phase5_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

negative_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Negative"), "count"].iat[0])
positive_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Positive"), "count"].iat[0])
neutral_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Neutral"), "count"].iat[0])
top_entity = entity_distribution_eda.iloc[0]
positive_top_words = ", ".join(top_tokens_lookup["Positive"].head(8)["token"].tolist())
negative_top_words = ", ".join(top_tokens_lookup["Negative"].head(8)["token"].tolist())
neutral_top_words = ", ".join(top_tokens_lookup["Neutral"].head(8)["token"].tolist())

phase5_insights_lines = [
    "# Phase 5 EDA Insights",
    "",
    f"- The cleaned dataset contains {len(df_eda):,} tweets across {df_eda['entity'].nunique():,} entities and three sentiment labels.",
    f"- Sentiment distribution is moderately imbalanced: Negative {negative_count:,}, Positive {positive_count:,}, and Neutral {neutral_count:,}.",
    f"- Negative is the largest class, so later modeling should use stratified splits and report macro F1 in addition to accuracy.",
    f"- The most frequent entity is {top_entity['entity']} with {int(top_entity['count']):,} rows ({float(top_entity['percent']):.2f}%). Entity counts are fairly spread across the 32 topics.",
    f"- Model token lengths are short enough for sequence modeling: p95 is {df_eda['model_token_count'].quantile(0.95):.0f} tokens and p99 is {df_eda['model_token_count'].quantile(0.99):.0f} tokens, supporting a later padding length of about 60.",
    f"- Top negative analysis tokens include: {negative_top_words}.",
    f"- Top positive analysis tokens include: {positive_top_words}.",
    f"- Top neutral analysis tokens include: {neutral_top_words}.",
    "- The duplicate-text audit from Phase 4 remains important for model splitting, because repeated cleaned text can inflate performance if similar examples appear in both train and test sets.",
    "",
    "## Generated Figures",
    "",
]
phase5_insights_lines.extend([f"- {name}: {path}" for name, path in phase5_figure_paths.items()])
phase5_insights_lines.extend(
    [
        "",
        "## Modeling Implications",
        "",
        "- Use stratified splitting to preserve sentiment proportions.",
        "- Build the model vocabulary only on the training split to avoid leakage.",
        "- Treat 60 tokens as the initial max sequence length candidate for RNN padding.",
        "- Consider deduplicating repeated model_text and sentiment pairs before final modeling experiments if validation leakage appears likely.",
    ]
)

phase5_insights_path = REPORTS_DIR / "phase5_eda_insights.md"
phase5_insights_path.write_text("\n".join(phase5_insights_lines) + "\n", encoding="utf-8")

print(f"Saved {len(phase5_tables)} Phase 5 EDA tables to {TABLES_DIR}")
print(f"Saved Phase 5 EDA insights to {phase5_insights_path}")

In [ ]:
# Render figures inline when the notebook is opened in Jupyter.
for figure_path in phase5_figure_paths.values():
    display_svg_file(figure_path)

## Phase 5 Findings To Carry Forward

EDA confirms that the cleaned three-class dataset is suitable for sequence modeling, with moderate class imbalance and short tweet lengths.

Carry-forward points:

- Use stratified train, validation, and test splits.
- Use macro F1 during evaluation because the classes are not perfectly balanced.
- Start RNN sequence padding at 60 tokens, based on the Phase 4 and Phase 5 token-length distributions.
- Build the vocabulary from training data only.
- Keep the Phase 4 duplicate-text audit in mind before final modeling, because repeated cleaned text can create optimistic test results if split randomly without care.

# Part 3: RNN Model Development

This part begins with feature engineering for the RNN pipeline. Phase 6 creates split-aware numerical inputs only; model training starts in the next phase.

## Phase 6: Feature Engineering

Phase 6 converts the Phase 4 preprocessed text into numerical artifacts for later RNN modeling without training a model.

Work completed in this phase:

- Load the Phase 4 preprocessed dataset.
- Create leakage-aware train, validation, and test splits by keeping identical `model_text` values in one split.
- Encode labels using the saved Phase 4 label mapping.
- Build the vocabulary from the training split only.
- Convert tweet tokens into padded integer sequences with a maximum length of 60.
- Save feature artifacts, split summaries, vocabulary metadata, and a concise Phase 6 report.

No embedding layer is trained in this phase. The saved vocabulary size and `padding_idx=0` setting are carried forward for the embedding layer in the RNN model architecture.

In [ ]:
# Phase 6 setup: load preprocessed text and modeling constants.
import json
from collections import Counter
import pandas as pd
import numpy as np

if "PROJECT_ROOT" not in globals():
    from pathlib import Path
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
    FIGURES_DIR = OUTPUT_DIR / "figures"
    MODELS_DIR = OUTPUT_DIR / "models"
    TABLES_DIR = OUTPUT_DIR / "tables"
    REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [DATA_OUTPUT_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(obj):
            print(obj)

PHASE4_PREPROCESSED_PATH = DATA_OUTPUT_DIR / "twitter_text_preprocessed_phase4.csv"
PHASE6_FEATURES_PATH = DATA_OUTPUT_DIR / "twitter_features_phase6.csv"
PHASE6_SEQUENCES_PATH = DATA_OUTPUT_DIR / "phase6_sequences.npz"
PHASE6_VOCAB_PATH = DATA_OUTPUT_DIR / "phase6_vocabulary.json"
PHASE6_TFIDF_VOCAB_PATH = DATA_OUTPUT_DIR / "phase6_tfidf_vocabulary.csv"
PHASE6_CONFIG_PATH = DATA_OUTPUT_DIR / "phase6_feature_config.json"

SPLIT_RANDOM_SEED = 42
SPLIT_PROPORTIONS = {"train": 0.70, "validation": 0.15, "test": 0.15}
MAX_SEQUENCE_LENGTH = 60
MAX_VOCAB_SIZE = 20_000
MIN_TOKEN_FREQUENCY = 2
TFIDF_FEATURE_COUNT = 5_000
PAD_TOKEN = "<PAD>"
OOV_TOKEN = "<OOV>"
PAD_ID = 0
OOV_ID = 1

required_phase6_columns = {
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "analysis_text",
    "raw_word_len",
    "model_token_count",
    "analysis_token_count",
}

df_features = pd.read_csv(PHASE4_PREPROCESSED_PATH)
missing_phase6_columns = sorted(required_phase6_columns - set(df_features.columns))
if missing_phase6_columns:
    raise ValueError(f"Missing expected Phase 6 columns: {missing_phase6_columns}")

phase4_label_mapping = pd.read_csv(TABLES_DIR / "phase4_label_mapping.csv")
label_to_id = dict(zip(phase4_label_mapping["sentiment"], phase4_label_mapping["label_id"]))
id_to_label = {int(label_id): label for label, label_id in label_to_id.items()}

expected_labels = {"Negative", "Neutral", "Positive"}
if set(df_features["sentiment"].unique()) != expected_labels:
    raise ValueError("Phase 6 expected only Negative, Neutral, and Positive labels.")

if sorted(label_to_id) != ["Negative", "Neutral", "Positive"]:
    raise ValueError(f"Unexpected Phase 4 label mapping: {label_to_id}")

df_features = df_features.reset_index(drop=False).rename(columns={"index": "source_row_id"})
df_features.insert(0, "phase6_row_id", np.arange(len(df_features), dtype=int))
df_features["model_text"] = df_features["model_text"].fillna("").astype(str)
df_features["model_tokens"] = df_features["model_text"].str.split()
df_features["label_id"] = df_features["sentiment"].map(label_to_id).astype(int)

empty_model_text_rows = int(df_features["model_text"].str.strip().eq("").sum())
if empty_model_text_rows:
    raise ValueError(f"Phase 6 found {empty_model_text_rows} empty model_text rows.")

print(f"Loaded Phase 4 preprocessed dataset: {PHASE4_PREPROCESSED_PATH}")
print(f"Feature engineering rows: {len(df_features):,}")
print(f"Label mapping: {label_to_id}")

In [ ]:
# Create a leakage-aware split by assigning each cleaned model_text group to one split.
def assign_group_preserving_splits(df, group_col, label_col, split_proportions, random_seed=42):
    split_names = list(split_proportions)
    proportions = np.array([split_proportions[name] for name in split_names], dtype=float)
    if not np.isclose(proportions.sum(), 1.0):
        raise ValueError("Split proportions must sum to 1.0")

    labels = sorted(df[label_col].unique())
    total_label_counts = df[label_col].value_counts().reindex(labels, fill_value=0).to_numpy(dtype=float)
    total_rows = float(len(df))
    target_label_counts = {
        name: total_label_counts * proportion
        for name, proportion in zip(split_names, proportions)
    }
    target_sizes = {
        name: total_rows * proportion
        for name, proportion in zip(split_names, proportions)
    }

    group_label_counts = (
        df.groupby([group_col, label_col])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    group_table = group_label_counts.reset_index()
    group_table["group_size"] = group_table[labels].sum(axis=1)
    group_table["dominant_label_count"] = group_table[labels].max(axis=1)

    rng = np.random.default_rng(random_seed)
    group_table["tie_breaker"] = rng.random(len(group_table))
    # Randomized group order avoids systematic text-length drift while the greedy
    # assignment below preserves split sizes and class balance.
    group_table = group_table.sort_values("tie_breaker")

    current_label_counts = {
        name: np.zeros(len(labels), dtype=float)
        for name in split_names
    }
    current_sizes = {name: 0.0 for name in split_names}
    assignments = {}

    def score_assignment(candidate_split, group_counts):
        score = 0.0
        for split_name in split_names:
            proposed_counts = current_label_counts[split_name].copy()
            proposed_size = current_sizes[split_name]
            if split_name == candidate_split:
                proposed_counts += group_counts
                proposed_size += float(group_counts.sum())

            label_error = np.sum(
                ((proposed_counts - target_label_counts[split_name]) / np.maximum(target_label_counts[split_name], 1.0)) ** 2
            )
            size_error = ((proposed_size - target_sizes[split_name]) / max(target_sizes[split_name], 1.0)) ** 2
            overfill_penalty = max(0.0, proposed_size - target_sizes[split_name]) / max(target_sizes[split_name], 1.0)
            score += label_error + size_error + (2.0 * overfill_penalty)
        return score

    for row in group_table.itertuples(index=False):
        group_value = getattr(row, group_col)
        group_counts = np.array([getattr(row, label) for label in labels], dtype=float)
        best_split = min(
            split_names,
            key=lambda name: (
                score_assignment(name, group_counts),
                current_sizes[name] / max(target_sizes[name], 1.0),
            ),
        )
        assignments[group_value] = best_split
        current_label_counts[best_split] += group_counts
        current_sizes[best_split] += float(group_counts.sum())

    audit_rows = []
    for split_name in split_names:
        row = {
            "split": split_name,
            "rows": int(current_sizes[split_name]),
            "target_rows": int(round(target_sizes[split_name])),
        }
        for label, count, target in zip(labels, current_label_counts[split_name], target_label_counts[split_name]):
            row[f"{label}_rows"] = int(count)
            row[f"{label}_target_rows"] = int(round(target))
        audit_rows.append(row)
    return assignments, pd.DataFrame(audit_rows)

split_assignments, group_split_audit = assign_group_preserving_splits(
    df_features,
    group_col="model_text",
    label_col="sentiment",
    split_proportions=SPLIT_PROPORTIONS,
    random_seed=SPLIT_RANDOM_SEED,
)

df_features["split"] = df_features["model_text"].map(split_assignments)
if df_features["split"].isna().any():
    raise ValueError("Some rows did not receive a split assignment.")

split_order = list(SPLIT_PROPORTIONS)
sentiment_order = ["Negative", "Neutral", "Positive"]
split_distribution = (
    df_features.groupby(["split", "sentiment"])
    .size()
    .rename("count")
    .reset_index()
)
split_distribution["split"] = pd.Categorical(split_distribution["split"], split_order, ordered=True)
split_distribution["sentiment"] = pd.Categorical(split_distribution["sentiment"], sentiment_order, ordered=True)
split_distribution = split_distribution.sort_values(["split", "sentiment"]).reset_index(drop=True)
split_totals = df_features["split"].value_counts().rename_axis("split").reset_index(name="split_rows")
split_distribution = split_distribution.merge(split_totals, on="split", how="left")
split_distribution["percent_within_split"] = (split_distribution["count"] / split_distribution["split_rows"] * 100).round(2)
split_distribution["percent_of_dataset"] = (split_distribution["count"] / len(df_features) * 100).round(2)

split_leakage_check = (
    df_features.groupby("model_text")["split"]
    .nunique()
    .gt(1)
    .sum()
)
conflicting_model_text_groups = int(df_features.groupby("model_text")["sentiment"].nunique().gt(1).sum())
rows_in_conflicting_model_text_groups = int(
    df_features.groupby("model_text").filter(lambda group: group["sentiment"].nunique() > 1).shape[0]
)

print(f"Assigned {df_features['model_text'].nunique():,} unique model_text groups to splits.")
print(f"model_text values appearing in multiple splits: {int(split_leakage_check):,}")
print(f"Conflicting-label model_text groups kept intact: {conflicting_model_text_groups:,}")
display(split_distribution)
display(group_split_audit)

In [ ]:
# Build the RNN vocabulary from training rows only.
train_mask = df_features["split"].eq("train")
train_token_counter = Counter(
    token
    for tokens in df_features.loc[train_mask, "model_tokens"]
    for token in tokens
)

eligible_vocab_tokens = [
    token
    for token, count in sorted(train_token_counter.items(), key=lambda item: (-item[1], item[0]))
    if count >= MIN_TOKEN_FREQUENCY
]
retained_vocab_tokens = eligible_vocab_tokens[: max(0, MAX_VOCAB_SIZE - 2)]

token_to_id = {PAD_TOKEN: PAD_ID, OOV_TOKEN: OOV_ID}
token_to_id.update({token: index + 2 for index, token in enumerate(retained_vocab_tokens)})
id_to_token = {index: token for token, index in token_to_id.items()}

vocabulary_summary = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "training_token_instances",
            "unique_training_tokens",
            "min_token_frequency",
            "max_vocab_size_including_special_tokens",
            "retained_text_tokens",
            "special_tokens",
            "final_vocabulary_size",
            "pad_token_id",
            "oov_token_id",
        ],
        "value": [
            f"{int(train_mask.sum()):,}",
            f"{sum(train_token_counter.values()):,}",
            f"{len(train_token_counter):,}",
            MIN_TOKEN_FREQUENCY,
            f"{MAX_VOCAB_SIZE:,}",
            f"{len(retained_vocab_tokens):,}",
            f"{PAD_TOKEN}, {OOV_TOKEN}",
            f"{len(token_to_id):,}",
            PAD_ID,
            OOV_ID,
        ],
    }
)

vocabulary_preview = pd.DataFrame(
    [
        {"token": token, "token_id": token_to_id[token], "train_count": train_token_counter[token]}
        for token in retained_vocab_tokens[:25]
    ]
)

print(f"Final vocabulary size: {len(token_to_id):,}")
print("Top training vocabulary tokens:")
display(vocabulary_preview)

In [ ]:
# Convert tokens to fixed-length integer sequences for RNN input.
def encode_and_pad_tokens(tokens, token_to_id, max_length):
    raw_ids = [token_to_id.get(token, OOV_ID) for token in tokens]
    clipped_ids = raw_ids[:max_length]
    padded_ids = clipped_ids + [PAD_ID] * (max_length - len(clipped_ids))
    sequence_length = len(clipped_ids)
    truncated_token_count = max(0, len(raw_ids) - max_length)
    oov_token_count = sum(1 for token_id in clipped_ids if token_id == OOV_ID)
    return padded_ids, sequence_length, truncated_token_count, oov_token_count

encoded_rows = [
    encode_and_pad_tokens(tokens, token_to_id, MAX_SEQUENCE_LENGTH)
    for tokens in df_features["model_tokens"]
]

sequence_matrix = np.array([row[0] for row in encoded_rows], dtype=np.int32)
df_features["sequence_length"] = [row[1] for row in encoded_rows]
df_features["truncated_token_count"] = [row[2] for row in encoded_rows]
df_features["oov_token_count"] = [row[3] for row in encoded_rows]
df_features["oov_token_rate"] = np.where(
    df_features["sequence_length"].gt(0),
    df_features["oov_token_count"] / df_features["sequence_length"],
    0.0,
).round(4)
df_features["sequence_ids"] = [" ".join(map(str, row)) for row in sequence_matrix]

labels_array = df_features["label_id"].to_numpy(dtype=np.int64)
row_ids_array = df_features["phase6_row_id"].to_numpy(dtype=np.int64)

sequence_arrays = {}
for split_name in split_order:
    indices = np.flatnonzero(df_features["split"].to_numpy() == split_name)
    sequence_arrays[f"X_{split_name}"] = sequence_matrix[indices]
    sequence_arrays[f"y_{split_name}"] = labels_array[indices]
    sequence_arrays[f"row_id_{split_name}"] = row_ids_array[indices]

np.savez_compressed(PHASE6_SEQUENCES_PATH, **sequence_arrays)

sequence_length_summary = (
    df_features.groupby("split")
    .agg(
        rows=("phase6_row_id", "count"),
        mean_model_tokens=("model_token_count", "mean"),
        p95_model_tokens=("model_token_count", lambda values: values.quantile(0.95)),
        max_model_tokens=("model_token_count", "max"),
        mean_sequence_length=("sequence_length", "mean"),
        rows_truncated=("truncated_token_count", lambda values: int((values > 0).sum())),
        total_truncated_tokens=("truncated_token_count", "sum"),
    )
    .round(2)
    .reset_index()
)
sequence_length_summary["split"] = pd.Categorical(sequence_length_summary["split"], split_order, ordered=True)
sequence_length_summary = sequence_length_summary.sort_values("split").reset_index(drop=True)

oov_summary_by_split = (
    df_features.groupby("split")
    .agg(
        rows=("phase6_row_id", "count"),
        mean_oov_token_count=("oov_token_count", "mean"),
        total_oov_tokens=("oov_token_count", "sum"),
        mean_oov_token_rate=("oov_token_rate", "mean"),
    )
    .round(4)
    .reset_index()
)
oov_summary_by_split["split"] = pd.Categorical(oov_summary_by_split["split"], split_order, ordered=True)
oov_summary_by_split = oov_summary_by_split.sort_values("split").reset_index(drop=True)

print(f"Sequence matrix shape: {sequence_matrix.shape}")
print(f"Saved compressed sequence arrays to: {PHASE6_SEQUENCES_PATH}")
display(sequence_length_summary)
display(oov_summary_by_split)

In [ ]:
# Build lightweight TF-IDF metadata from the training split for a later baseline comparison.
tfidf_tokens = retained_vocab_tokens[: min(TFIDF_FEATURE_COUNT, len(retained_vocab_tokens))]
tfidf_token_set = set(tfidf_tokens)
train_documents = df_features.loc[train_mask, "model_tokens"].tolist()
train_document_count = len(train_documents)

document_frequency = Counter()
for tokens in train_documents:
    document_frequency.update(set(token for token in tokens if token in tfidf_token_set))

tfidf_vocab = pd.DataFrame(
    [
        {
            "tfidf_feature_id": feature_id,
            "token": token,
            "token_id": token_to_id[token],
            "train_document_frequency": int(document_frequency[token]),
            "idf": round(float(np.log((1 + train_document_count) / (1 + document_frequency[token])) + 1), 6),
            "train_token_count": int(train_token_counter[token]),
        }
        for feature_id, token in enumerate(tfidf_tokens)
    ]
)

tfidf_top_terms_rows = []
idf_lookup = dict(zip(tfidf_vocab["token"], tfidf_vocab["idf"]))
for sentiment in sentiment_order:
    sentiment_scores = Counter()
    sentiment_docs = df_features.loc[train_mask & df_features["sentiment"].eq(sentiment), "model_tokens"]
    doc_count = len(sentiment_docs)
    for tokens in sentiment_docs:
        counts = Counter(token for token in tokens if token in tfidf_token_set)
        token_total = sum(counts.values()) or 1
        for token, count in counts.items():
            sentiment_scores[token] += (count / token_total) * idf_lookup[token]
    for rank, (token, score) in enumerate(sentiment_scores.most_common(20), start=1):
        tfidf_top_terms_rows.append(
            {
                "sentiment": sentiment,
                "rank": rank,
                "token": token,
                "mean_train_tfidf": round(score / max(doc_count, 1), 6),
                "token_id": token_to_id[token],
                "idf": idf_lookup[token],
            }
        )

phase6_tfidf_top_terms_by_sentiment = pd.DataFrame(tfidf_top_terms_rows)

tfidf_vocab.to_csv(PHASE6_TFIDF_VOCAB_PATH, index=False)

print(f"Saved TF-IDF vocabulary metadata with {len(tfidf_vocab):,} features to: {PHASE6_TFIDF_VOCAB_PATH}")
display(phase6_tfidf_top_terms_by_sentiment.groupby("sentiment").head(8))

In [ ]:
# Save Phase 6 artifacts, tables, and report summary.
phase6_feature_columns = [
    "phase6_row_id",
    "source_row_id",
    "split",
    "tweet_id",
    "entity",
    "sentiment",
    "label_id",
    "model_text",
    "model_token_count",
    "sequence_length",
    "truncated_token_count",
    "oov_token_count",
    "oov_token_rate",
    "sequence_ids",
]
df_features[phase6_feature_columns].to_csv(PHASE6_FEATURES_PATH, index=False)

vocab_payload = {
    "created_from": "training split only",
    "max_vocab_size": MAX_VOCAB_SIZE,
    "min_token_frequency": MIN_TOKEN_FREQUENCY,
    "special_tokens": {"pad": PAD_TOKEN, "oov": OOV_TOKEN},
    "pad_token_id": PAD_ID,
    "oov_token_id": OOV_ID,
    "vocabulary_size": len(token_to_id),
    "token_to_id": token_to_id,
}
PHASE6_VOCAB_PATH.write_text(json.dumps(vocab_payload, indent=2, sort_keys=True), encoding="utf-8")

phase6_config = {
    "random_seed": SPLIT_RANDOM_SEED,
    "split_proportions": SPLIT_PROPORTIONS,
    "group_column_for_split": "model_text",
    "label_to_id": {label: int(label_id) for label, label_id in label_to_id.items()},
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "max_vocab_size": MAX_VOCAB_SIZE,
    "min_token_frequency": MIN_TOKEN_FREQUENCY,
    "pad_token": PAD_TOKEN,
    "pad_token_id": PAD_ID,
    "oov_token": OOV_TOKEN,
    "oov_token_id": OOV_ID,
    "embedding_input_dim_for_phase7": len(token_to_id),
    "embedding_padding_idx_for_phase7": PAD_ID,
    "tfidf_feature_count": len(tfidf_vocab),
    "model_training_run": False,
}
PHASE6_CONFIG_PATH.write_text(json.dumps(phase6_config, indent=2), encoding="utf-8")

phase6_sample_sequences = df_features[
    [
        "phase6_row_id",
        "split",
        "sentiment",
        "label_id",
        "model_text",
        "sequence_length",
        "oov_token_count",
        "sequence_ids",
    ]
].sample(12, random_state=SPLIT_RANDOM_SEED).copy()
phase6_sample_sequences["model_tokens_preview"] = phase6_sample_sequences["model_text"].str.split().apply(lambda tokens: " ".join(tokens[:20]))
phase6_sample_sequences["sequence_ids_preview"] = phase6_sample_sequences["sequence_ids"].str.split().apply(lambda ids: " ".join(ids[:20]))
phase6_sample_sequences = phase6_sample_sequences.drop(columns=["sequence_ids"])

phase6_tables = {
    "phase6_split_distribution.csv": split_distribution,
    "phase6_group_split_audit.csv": group_split_audit,
    "phase6_vocabulary_summary.csv": vocabulary_summary,
    "phase6_sequence_length_summary.csv": sequence_length_summary,
    "phase6_oov_summary_by_split.csv": oov_summary_by_split,
    "phase6_tfidf_top_terms_by_sentiment.csv": phase6_tfidf_top_terms_by_sentiment,
    "phase6_sample_sequences.csv": phase6_sample_sequences,
}
for filename, table in phase6_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

train_rows = int(df_features["split"].eq("train").sum())
validation_rows = int(df_features["split"].eq("validation").sum())
test_rows = int(df_features["split"].eq("test").sum())
rows_truncated = int(df_features["truncated_token_count"].gt(0).sum())
model_text_values_in_multiple_splits = int(
    df_features.groupby("model_text")["split"].nunique().gt(1).sum()
)

phase6_summary_lines = [
    "# Phase 6 Feature Engineering Summary",
    "",
    f"- Source preprocessed dataset: {PHASE4_PREPROCESSED_PATH}",
    f"- Feature dataset: {PHASE6_FEATURES_PATH}",
    f"- Sequence arrays: {PHASE6_SEQUENCES_PATH}",
    f"- Vocabulary metadata: {PHASE6_VOCAB_PATH}",
    f"- TF-IDF vocabulary metadata: {PHASE6_TFIDF_VOCAB_PATH}",
    f"- Rows engineered: {len(df_features):,}",
    f"- Split rows: train {train_rows:,}, validation {validation_rows:,}, test {test_rows:,}",
    f"- Split strategy: group-preserving split on model_text to reduce duplicate-text leakage.",
    f"- model_text values appearing in multiple splits: {model_text_values_in_multiple_splits:,}",
    f"- Conflicting-label model_text groups kept in a single split: {conflicting_model_text_groups:,}",
    f"- Rows in conflicting-label model_text groups: {rows_in_conflicting_model_text_groups:,}",
    f"- Label mapping: Negative -> {label_to_id['Negative']}, Neutral -> {label_to_id['Neutral']}, Positive -> {label_to_id['Positive']}",
    f"- Max sequence length: {MAX_SEQUENCE_LENGTH}",
    f"- Rows truncated at max sequence length: {rows_truncated:,}",
    f"- Vocabulary built from training split only: {len(token_to_id):,} tokens including {PAD_TOKEN} and {OOV_TOKEN}.",
    f"- Embedding carry-forward: input_dim={len(token_to_id):,}, padding_idx={PAD_ID}.",
    f"- TF-IDF carry-forward: {len(tfidf_vocab):,} training-derived features saved for an optional baseline.",
    "- Model training run in this phase: no.",
    "",
    "## Artifacts",
    "",
]
phase6_summary_lines.extend([f"- {name}: {TABLES_DIR / name}" for name in phase6_tables])
phase6_summary_lines.extend(
    [
        "",
        "## Carry-Forward Notes",
        "",
        "- Use phase6_sequences.npz for PyTorch Dataset objects in Phase 7.",
        "- Keep PAD_ID as 0 and pass padding_idx=0 to the embedding layer.",
        "- Use the validation split for model selection and the test split only for final evaluation.",
        "- The vocabulary and TF-IDF metadata were fit on training rows only, so later modeling should reuse these artifacts instead of refitting on validation or test data.",
    ]
)

phase6_summary_path = REPORTS_DIR / "phase6_feature_engineering_summary.md"
phase6_summary_path.write_text("\n".join(phase6_summary_lines) + "\n", encoding="utf-8")

print(f"Saved Phase 6 feature dataset to {PHASE6_FEATURES_PATH}")
print(f"Saved {len(phase6_tables)} Phase 6 tables to {TABLES_DIR}")
print(f"Saved Phase 6 summary to {phase6_summary_path}")
print("No model training was run.")

## Phase 6 Findings To Carry Forward

Phase 6 prepares the modeling inputs without training an RNN.

Carry-forward points:

- Use `outputs/data/phase6_sequences.npz` for train, validation, and test arrays in Phase 7.
- Use `outputs/data/phase6_vocabulary.json` to keep token IDs stable.
- Keep `padding_idx=0` in the PyTorch embedding layer because `<PAD>` is encoded as `0`.
- Use the validation split for model selection and reserve the test split for final evaluation.
- Reuse the saved train-fitted vocabulary and TF-IDF metadata; do not refit them on validation or test rows.

## Phase 7: Baseline RNN Modeling

Phase 7 trains the first embedding-based recurrent neural network using the Phase 6 train and validation splits.

Work completed in this phase:

- Load the padded sequence arrays and feature configuration from Phase 6.
- Build PyTorch `Dataset` and `DataLoader` objects.
- Define a compact embedding + GRU classifier with dropout.
- Train only on the training split and monitor validation performance after each epoch.
- Save the best validation checkpoint, training history, validation metrics, and a learning-curve figure.

The held-out test split remains unused in this phase. Final test-set evaluation belongs to Phase 8.

In [ ]:
# Phase 7 setup: imports, paths, and reproducibility.
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
    FIGURES_DIR = OUTPUT_DIR / "figures"
    MODELS_DIR = OUTPUT_DIR / "models"
    TABLES_DIR = OUTPUT_DIR / "tables"
    REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [FIGURES_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(obj):
            print(obj)

PHASE6_SEQUENCES_PATH = DATA_OUTPUT_DIR / "phase6_sequences.npz"
PHASE6_CONFIG_PATH = DATA_OUTPUT_DIR / "phase6_feature_config.json"
PHASE7_CHECKPOINT_PATH = MODELS_DIR / "phase7_baseline_gru_state.pt"
PHASE7_MODEL_METADATA_PATH = MODELS_DIR / "phase7_baseline_gru_metadata.json"
PHASE7_HISTORY_PATH = TABLES_DIR / "phase7_training_history.csv"
PHASE7_VALIDATION_METRICS_PATH = TABLES_DIR / "phase7_validation_metrics.csv"
PHASE7_CONFUSION_MATRIX_PATH = TABLES_DIR / "phase7_validation_confusion_matrix.csv"
PHASE7_MODEL_CONFIG_PATH = TABLES_DIR / "phase7_model_config.csv"
PHASE7_LEARNING_CURVE_PATH = FIGURES_DIR / "phase7_learning_curve.svg"
PHASE7_SUMMARY_PATH = REPORTS_DIR / "phase7_rnn_modeling_summary.md"

PHASE7_RANDOM_SEED = 42
random.seed(PHASE7_RANDOM_SEED)
np.random.seed(PHASE7_RANDOM_SEED)
torch.manual_seed(PHASE7_RANDOM_SEED)
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Training device: {DEVICE}")
print(f"Torch CPU threads: {torch.get_num_threads()}")

In [ ]:
# Load Phase 6 sequence arrays and configuration.
with np.load(PHASE6_SEQUENCES_PATH) as sequence_data:
    X_train = sequence_data["X_train"].astype(np.int64)
    y_train = sequence_data["y_train"].astype(np.int64)
    X_validation = sequence_data["X_validation"].astype(np.int64)
    y_validation = sequence_data["y_validation"].astype(np.int64)
    # The test split is loaded only for shape documentation and remains unused for model selection.
    X_test_shape = tuple(sequence_data["X_test"].shape)
    y_test_shape = tuple(sequence_data["y_test"].shape)

phase6_config = json.loads(PHASE6_CONFIG_PATH.read_text(encoding="utf-8"))
label_to_id = {label: int(label_id) for label, label_id in phase6_config["label_to_id"].items()}
id_to_label = {label_id: label for label, label_id in label_to_id.items()}
class_names = [id_to_label[index] for index in sorted(id_to_label)]

vocab_size = int(phase6_config["embedding_input_dim_for_phase7"])
padding_idx = int(phase6_config["embedding_padding_idx_for_phase7"])
max_sequence_length = int(phase6_config["max_sequence_length"])
num_classes = len(class_names)

train_lengths = np.maximum((X_train != padding_idx).sum(axis=1), 1).astype(np.int64)
validation_lengths = np.maximum((X_validation != padding_idx).sum(axis=1), 1).astype(np.int64)

print(f"Train arrays: X={X_train.shape}, y={y_train.shape}")
print(f"Validation arrays: X={X_validation.shape}, y={y_validation.shape}")
print(f"Held-out test arrays reserved for Phase 8: X={X_test_shape}, y={y_test_shape}")
print(f"Vocabulary size: {vocab_size:,}; max sequence length: {max_sequence_length}; classes: {class_names}")

In [ ]:
# Build DataLoaders and baseline GRU model.
BATCH_SIZE = 512
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
NUM_LAYERS = 1
DROPOUT_RATE = 0.30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 3
GRADIENT_CLIP_NORM = 1.0

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(train_lengths),
    torch.from_numpy(y_train),
)
validation_dataset = TensorDataset(
    torch.from_numpy(X_validation),
    torch.from_numpy(validation_lengths),
    torch.from_numpy(y_validation),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class GRUSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, padding_idx, dropout_rate=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout_rate)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        embedded = self.dropout(self.embedding(input_ids))
        lengths_cpu = lengths.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths_cpu,
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.gru(packed)
        final_hidden = self.dropout(hidden[-1])
        return self.classifier(final_hidden)

model = GRUSentimentClassifier(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=num_classes,
    padding_idx=padding_idx,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

class_counts = np.bincount(y_train, minlength=num_classes)
class_weights = (len(y_train) / (num_classes * np.maximum(class_counts, 1))).astype(np.float32)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

phase7_model_config = {
    "model_name": "baseline_embedding_gru",
    "vocab_size": vocab_size,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout_rate": DROPOUT_RATE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "class_weights": {class_names[index]: float(class_weights[index]) for index in range(num_classes)},
    "parameter_count": int(parameter_count),
    "trainable_parameter_count": int(trainable_parameter_count),
    "device": str(DEVICE),
    "test_split_used": False,
}

print(f"Model parameters: {parameter_count:,} total; {trainable_parameter_count:,} trainable")
print(f"Training batches: {len(train_loader):,}; validation batches: {len(validation_loader):,}")
display(pd.DataFrame([phase7_model_config]).T.reset_index().rename(columns={"index": "setting", 0: "value"}))

In [ ]:
# Training and validation helpers.
def update_confusion_matrix(confusion_matrix, true_labels, predicted_labels):
    for true_label, predicted_label in zip(true_labels, predicted_labels):
        confusion_matrix[int(true_label), int(predicted_label)] += 1


def classification_metrics_from_confusion(confusion_matrix, class_names):
    rows = []
    total = confusion_matrix.sum()
    accuracy = float(np.trace(confusion_matrix) / total) if total else 0.0
    for index, class_name in enumerate(class_names):
        true_positive = float(confusion_matrix[index, index])
        false_positive = float(confusion_matrix[:, index].sum() - true_positive)
        false_negative = float(confusion_matrix[index, :].sum() - true_positive)
        support = float(confusion_matrix[index, :].sum())
        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append(
            {
                "label": class_name,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "support": int(support),
            }
        )
    metrics = pd.DataFrame(rows)
    summary_rows = pd.DataFrame(
        [
            {
                "label": "accuracy",
                "precision": accuracy,
                "recall": accuracy,
                "f1_score": accuracy,
                "support": int(total),
            },
            {
                "label": "macro_avg",
                "precision": metrics["precision"].mean(),
                "recall": metrics["recall"].mean(),
                "f1_score": metrics["f1_score"].mean(),
                "support": int(total),
            },
            {
                "label": "weighted_avg",
                "precision": np.average(metrics["precision"], weights=metrics["support"]),
                "recall": np.average(metrics["recall"], weights=metrics["support"]),
                "f1_score": np.average(metrics["f1_score"], weights=metrics["support"]),
                "support": int(total),
            },
        ]
    )
    return pd.concat([metrics, summary_rows], ignore_index=True)


def run_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_examples = 0
    correct = 0
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

    for input_ids, lengths, labels in data_loader:
        input_ids = input_ids.to(DEVICE)
        lengths = lengths.to(DEVICE)
        labels = labels.to(DEVICE)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            if is_training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
        correct += int((predictions == labels).sum().item())
        update_confusion_matrix(
            confusion_matrix,
            labels.detach().cpu().numpy(),
            predictions.detach().cpu().numpy(),
        )

    average_loss = total_loss / max(total_examples, 1)
    accuracy = correct / max(total_examples, 1)
    metrics_table = classification_metrics_from_confusion(confusion_matrix, class_names)
    macro_f1 = float(metrics_table.loc[metrics_table["label"].eq("macro_avg"), "f1_score"].iat[0])
    return average_loss, accuracy, macro_f1, confusion_matrix, metrics_table

print("Phase 7 training helpers ready.")

In [ ]:
# Train the baseline GRU and keep the best validation checkpoint.
history_rows = []
best_validation_macro_f1 = -1.0
best_epoch = None
best_confusion_matrix = None
best_validation_metrics = None
training_started_at = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started_at = time.perf_counter()
    train_loss, train_accuracy, train_macro_f1, _, _ = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
    )
    validation_loss, validation_accuracy, validation_macro_f1, validation_confusion_matrix, validation_metrics = run_epoch(
        model,
        validation_loader,
        criterion,
        optimizer=None,
    )
    epoch_seconds = time.perf_counter() - epoch_started_at

    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "train_macro_f1": train_macro_f1,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
            "validation_macro_f1": validation_macro_f1,
            "epoch_seconds": round(epoch_seconds, 2),
        }
    )

    if validation_macro_f1 > best_validation_macro_f1:
        best_validation_macro_f1 = validation_macro_f1
        best_epoch = epoch
        best_confusion_matrix = validation_confusion_matrix.copy()
        best_validation_metrics = validation_metrics.copy()
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_config": phase7_model_config,
                "phase6_config": phase6_config,
                "class_names": class_names,
                "best_epoch": best_epoch,
                "best_validation_macro_f1": best_validation_macro_f1,
            },
            PHASE7_CHECKPOINT_PATH,
        )

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_accuracy:.4f}, macro F1 {train_macro_f1:.4f} | "
        f"val loss {validation_loss:.4f}, acc {validation_accuracy:.4f}, macro F1 {validation_macro_f1:.4f} | "
        f"{epoch_seconds:.1f}s"
    )

total_training_seconds = time.perf_counter() - training_started_at
phase7_training_history = pd.DataFrame(history_rows)

print(f"Best validation macro F1: {best_validation_macro_f1:.4f} at epoch {best_epoch}")
print(f"Total Phase 7 training time: {total_training_seconds:.1f}s")
display(phase7_training_history)

In [ ]:
# Save Phase 7 tables, metadata, and learning-curve SVG.
def svg_text(text):
    import html
    return html.escape(str(text), quote=True)


def save_learning_curve_svg(history, path, width=900, height=520):
    margin_left, margin_right, margin_top, margin_bottom = 80, 150, 70, 70
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    epochs = history["epoch"].to_numpy(dtype=float)
    series = {
        "train_loss": (history["train_loss"].to_numpy(dtype=float), "#b4494b"),
        "validation_loss": (history["validation_loss"].to_numpy(dtype=float), "#2f6f9f"),
        "train_macro_f1": (history["train_macro_f1"].to_numpy(dtype=float), "#8a6f2a"),
        "validation_macro_f1": (history["validation_macro_f1"].to_numpy(dtype=float), "#3f8f61"),
    }
    y_values = np.concatenate([values for values, _ in series.values()])
    y_min = max(0.0, float(y_values.min()) - 0.05)
    y_max = min(1.5, float(y_values.max()) + 0.05)
    if y_max <= y_min:
        y_max = y_min + 1.0

    def x_for_epoch(epoch):
        if len(epochs) == 1:
            return margin_left + plot_width / 2
        return margin_left + (epoch - epochs.min()) / (epochs.max() - epochs.min()) * plot_width

    def y_for_value(value):
        return margin_top + (y_max - value) / (y_max - y_min) * plot_height

    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Phase 7 Baseline GRU Learning Curve</text>',
        f'<line x1="{margin_left}" y1="{height-margin_bottom}" x2="{width-margin_right}" y2="{height-margin_bottom}" stroke="#333"/>',
        f'<line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{height-margin_bottom}" stroke="#333"/>',
        f'<text x="{width/2}" y="{height-24}" text-anchor="middle" font-family="Arial" font-size="14" fill="#333">Epoch</text>',
    ]
    for tick in np.linspace(y_min, y_max, 6):
        y = y_for_value(tick)
        parts.append(f'<line x1="{margin_left}" y1="{y:.1f}" x2="{width-margin_right}" y2="{y:.1f}" stroke="#e5e5e5"/>')
        parts.append(f'<text x="{margin_left-10}" y="{y+4:.1f}" text-anchor="end" font-family="Arial" font-size="12" fill="#555">{tick:.2f}</text>')
    for epoch in epochs:
        x = x_for_epoch(epoch)
        parts.append(f'<text x="{x:.1f}" y="{height-margin_bottom+24}" text-anchor="middle" font-family="Arial" font-size="12" fill="#555">{int(epoch)}</text>')
    legend_y = margin_top + 8
    for label_index, (label, (values, color)) in enumerate(series.items()):
        points = " ".join([f"{x_for_epoch(epoch):.1f},{y_for_value(value):.1f}" for epoch, value in zip(epochs, values)])
        parts.append(f'<polyline fill="none" stroke="{color}" stroke-width="3" points="{points}"/>')
        for epoch, value in zip(epochs, values):
            parts.append(f'<circle cx="{x_for_epoch(epoch):.1f}" cy="{y_for_value(value):.1f}" r="4" fill="{color}"/>')
        y = legend_y + label_index * 24
        parts.append(f'<rect x="{width-margin_right+25}" y="{y-10}" width="14" height="14" fill="{color}"/>')
        parts.append(f'<text x="{width-margin_right+46}" y="{y+2}" font-family="Arial" font-size="13" fill="#222">{svg_text(label)}</text>')
    path.write_text(
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img">\n'
        + "\n".join(parts)
        + "\n</svg>\n",
        encoding="utf-8",
    )

phase7_training_history.to_csv(PHASE7_HISTORY_PATH, index=False)
best_validation_metrics = best_validation_metrics.copy()
for metric_column in ["precision", "recall", "f1_score"]:
    best_validation_metrics[metric_column] = best_validation_metrics[metric_column].round(4)
best_validation_metrics.to_csv(PHASE7_VALIDATION_METRICS_PATH, index=False)

phase7_confusion_matrix = pd.DataFrame(best_confusion_matrix, index=class_names, columns=class_names)
phase7_confusion_matrix.index.name = "actual_label"
phase7_confusion_matrix.to_csv(PHASE7_CONFUSION_MATRIX_PATH)

phase7_model_config_table = pd.DataFrame(
    [{"setting": key, "value": json.dumps(value) if isinstance(value, (dict, list)) else value} for key, value in phase7_model_config.items()]
)
phase7_model_config_table.to_csv(PHASE7_MODEL_CONFIG_PATH, index=False)

phase7_metadata = {
    "checkpoint_path": str(PHASE7_CHECKPOINT_PATH),
    "history_path": str(PHASE7_HISTORY_PATH),
    "best_epoch": int(best_epoch),
    "best_validation_macro_f1": float(best_validation_macro_f1),
    "total_training_seconds": round(float(total_training_seconds), 2),
    "model_config": phase7_model_config,
    "class_names": class_names,
    "test_split_used": False,
}
PHASE7_MODEL_METADATA_PATH.write_text(json.dumps(phase7_metadata, indent=2), encoding="utf-8")
save_learning_curve_svg(phase7_training_history, PHASE7_LEARNING_CURVE_PATH)

print(f"Saved checkpoint to {PHASE7_CHECKPOINT_PATH}")
print(f"Saved training history to {PHASE7_HISTORY_PATH}")
print(f"Saved validation metrics to {PHASE7_VALIDATION_METRICS_PATH}")
print(f"Saved learning curve to {PHASE7_LEARNING_CURVE_PATH}")
display(best_validation_metrics)
display(phase7_confusion_matrix)

In [ ]:
# Save concise Phase 7 report summary.
best_history_row = phase7_training_history.loc[phase7_training_history["epoch"].eq(best_epoch)].iloc[0]
macro_row = best_validation_metrics.loc[best_validation_metrics["label"].eq("macro_avg")].iloc[0]
accuracy_row = best_validation_metrics.loc[best_validation_metrics["label"].eq("accuracy")].iloc[0]

phase7_summary_lines = [
    "# Phase 7 RNN Modeling Summary",
    "",
    f"- Source sequence arrays: {PHASE6_SEQUENCES_PATH}",
    f"- Model checkpoint: {PHASE7_CHECKPOINT_PATH}",
    f"- Model metadata: {PHASE7_MODEL_METADATA_PATH}",
    f"- Training history: {PHASE7_HISTORY_PATH}",
    f"- Validation metrics: {PHASE7_VALIDATION_METRICS_PATH}",
    f"- Validation confusion matrix: {PHASE7_CONFUSION_MATRIX_PATH}",
    f"- Learning curve: {PHASE7_LEARNING_CURVE_PATH}",
    f"- Model: embedding + GRU baseline with dropout.",
    f"- Vocabulary size: {vocab_size:,}",
    f"- Embedding dimension: {EMBEDDING_DIM}",
    f"- Hidden dimension: {HIDDEN_DIM}",
    f"- Train rows: {len(y_train):,}",
    f"- Validation rows: {len(y_validation):,}",
    f"- Epochs trained: {MAX_EPOCHS}",
    f"- Best epoch by validation macro F1: {int(best_epoch)}",
    f"- Best validation loss: {best_history_row['validation_loss']:.4f}",
    f"- Best validation accuracy: {float(accuracy_row['f1_score']):.4f}",
    f"- Best validation macro F1: {float(macro_row['f1_score']):.4f}",
    f"- Total training time: {total_training_seconds:.1f} seconds",
    "- Held-out test split used: no.",
    "",
    "## Carry-Forward Notes",
    "",
    "- Phase 8 should load the saved checkpoint and evaluate once on the held-out test split.",
    "- Report macro F1 alongside accuracy because the classes are moderately imbalanced.",
    "- Phase 9 can compare this compact GRU baseline against larger GRU/LSTM variants or tuned dropout/hidden dimensions.",
]
PHASE7_SUMMARY_PATH.write_text("\n".join(phase7_summary_lines) + "\n", encoding="utf-8")

print(f"Saved Phase 7 summary to {PHASE7_SUMMARY_PATH}")

## Phase 7 Findings To Carry Forward

Phase 7 trains the first baseline RNN and saves the best validation checkpoint.

Carry-forward points:

- Use `outputs/models/phase7_baseline_gru_state.pt` as the baseline checkpoint for Phase 8 evaluation.
- Use `outputs/tables/phase7_training_history.csv` for learning-curve reporting.
- Use `outputs/tables/phase7_validation_metrics.csv` and `outputs/tables/phase7_validation_confusion_matrix.csv` as validation-only model-selection evidence.
- Keep the held-out test split untouched until Phase 8.

# Part 4: Evaluation, Improvement, And Presentation

This part begins with final held-out evaluation of the Phase 7 baseline model. Model improvement and presentation-ready synthesis follow in later phases.

## Phase 8: Held-Out Test Evaluation

Phase 8 evaluates the saved Phase 7 baseline GRU exactly once on the untouched test split.

Work completed in this phase:

- Load the Phase 7 checkpoint and metadata.
- Recreate the embedding + GRU architecture from saved model configuration.
- Load `X_test` and `y_test` from the Phase 6 sequence artifact.
- Compute test accuracy, precision, recall, macro F1, weighted F1, and confusion matrix.
- Save full test predictions, sample predictions, evaluation tables, confusion-matrix figure, and a concise report.

This phase does not retrain, tune, or select a model. It uses the held-out test split only for final baseline evaluation.

In [ ]:
# Phase 8 setup: imports, paths, and deterministic evaluation mode.
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
    FIGURES_DIR = OUTPUT_DIR / "figures"
    MODELS_DIR = OUTPUT_DIR / "models"
    TABLES_DIR = OUTPUT_DIR / "tables"
    REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [DATA_OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(obj):
            print(obj)

PHASE6_SEQUENCES_PATH = DATA_OUTPUT_DIR / "phase6_sequences.npz"
PHASE6_FEATURES_PATH = DATA_OUTPUT_DIR / "twitter_features_phase6.csv"
PHASE7_CHECKPOINT_PATH = MODELS_DIR / "phase7_baseline_gru_state.pt"
PHASE7_MODEL_METADATA_PATH = MODELS_DIR / "phase7_baseline_gru_metadata.json"

PHASE8_FULL_PREDICTIONS_PATH = DATA_OUTPUT_DIR / "phase8_test_predictions.csv"
PHASE8_TEST_METRICS_PATH = TABLES_DIR / "phase8_test_metrics.csv"
PHASE8_CONFUSION_MATRIX_PATH = TABLES_DIR / "phase8_test_confusion_matrix.csv"
PHASE8_CONFUSION_MATRIX_NORMALIZED_PATH = TABLES_DIR / "phase8_test_confusion_matrix_normalized.csv"
PHASE8_PREDICTION_DISTRIBUTION_PATH = TABLES_DIR / "phase8_test_prediction_distribution.csv"
PHASE8_SAMPLE_PREDICTIONS_PATH = TABLES_DIR / "phase8_test_prediction_samples.csv"
PHASE8_EVALUATION_OVERVIEW_PATH = TABLES_DIR / "phase8_evaluation_overview.csv"
PHASE8_CONFUSION_MATRIX_FIGURE_PATH = FIGURES_DIR / "phase8_test_confusion_matrix.svg"
PHASE8_SUMMARY_PATH = REPORTS_DIR / "phase8_evaluation_summary.md"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))

print(f"PyTorch version: {torch.__version__}")
print(f"Evaluation device: {DEVICE}")

In [ ]:
# Load checkpoint, metadata, test arrays, and row-level feature references.
checkpoint = torch.load(PHASE7_CHECKPOINT_PATH, map_location=DEVICE)
phase7_metadata = json.loads(PHASE7_MODEL_METADATA_PATH.read_text(encoding="utf-8"))
model_config = checkpoint["model_config"]
phase6_config = checkpoint["phase6_config"]
class_names = checkpoint["class_names"]
label_to_id = {label: int(label_id) for label, label_id in phase6_config["label_to_id"].items()}
id_to_label = {label_id: label for label, label_id in label_to_id.items()}

with np.load(PHASE6_SEQUENCES_PATH) as sequence_data:
    X_test = sequence_data["X_test"].astype(np.int64)
    y_test = sequence_data["y_test"].astype(np.int64)
    row_id_test = sequence_data["row_id_test"].astype(np.int64)

features_lookup = pd.read_csv(
    PHASE6_FEATURES_PATH,
    usecols=["phase6_row_id", "tweet_id", "entity", "sentiment", "model_text", "model_token_count", "split"],
)
test_feature_rows = (
    pd.DataFrame({"phase6_row_id": row_id_test})
    .merge(features_lookup, on="phase6_row_id", how="left", validate="one_to_one")
)

if test_feature_rows["split"].isna().any():
    raise ValueError("Some test row IDs were not found in the Phase 6 feature audit.")
if not test_feature_rows["split"].eq("test").all():
    raise ValueError("Phase 8 expected every evaluated row to come from the Phase 6 test split.")
if not np.array_equal(y_test, test_feature_rows["sentiment"].map(label_to_id).to_numpy(dtype=np.int64)):
    raise ValueError("Test labels in sequence arrays do not match the Phase 6 feature audit.")

padding_idx = int(model_config.get("padding_idx", phase6_config["embedding_padding_idx_for_phase7"]))
test_lengths = np.maximum((X_test != padding_idx).sum(axis=1), 1).astype(np.int64)

print(f"Loaded checkpoint: {PHASE7_CHECKPOINT_PATH}")
print(f"Best validation macro F1 from Phase 7: {checkpoint['best_validation_macro_f1']:.4f}")
print(f"Test arrays: X={X_test.shape}, y={y_test.shape}")
print(f"Test feature rows verified: {len(test_feature_rows):,}")

In [ ]:
# Recreate the Phase 7 GRU model and load the saved weights.
class GRUSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, padding_idx, dropout_rate=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout_rate)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        embedded = self.dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, hidden = self.gru(packed)
        final_hidden = self.dropout(hidden[-1])
        return self.classifier(final_hidden)

model = GRUSentimentClassifier(
    vocab_size=int(model_config["vocab_size"]),
    embedding_dim=int(model_config["embedding_dim"]),
    hidden_dim=int(model_config["hidden_dim"]),
    num_classes=len(class_names),
    padding_idx=padding_idx,
    dropout_rate=float(model_config["dropout_rate"]),
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Restored model parameters: {parameter_count:,}")
print(f"Class order: {class_names}")

In [ ]:
# Evaluate exactly once on the held-out test split.
BATCH_SIZE = int(model_config.get("batch_size", 512))
test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(test_lengths),
    torch.from_numpy(y_test),
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class_weights = torch.tensor(
    [float(model_config["class_weights"][class_name]) for class_name in class_names],
    dtype=torch.float32,
    device=DEVICE,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

all_probabilities = []
all_predictions = []
total_loss = 0.0
total_examples = 0

with torch.no_grad():
    for input_ids, lengths, labels in test_loader:
        input_ids = input_ids.to(DEVICE)
        lengths = lengths.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size
        all_probabilities.append(probabilities.detach().cpu().numpy())
        all_predictions.append(predictions.detach().cpu().numpy())

test_probabilities = np.vstack(all_probabilities)
test_predictions = np.concatenate(all_predictions)
test_loss = total_loss / max(total_examples, 1)

def build_confusion_matrix(true_labels, predicted_labels, num_classes):
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for true_label, predicted_label in zip(true_labels, predicted_labels):
        confusion_matrix[int(true_label), int(predicted_label)] += 1
    return confusion_matrix


def classification_metrics_from_confusion(confusion_matrix, class_names):
    rows = []
    total = confusion_matrix.sum()
    accuracy = float(np.trace(confusion_matrix) / total) if total else 0.0
    for index, class_name in enumerate(class_names):
        true_positive = float(confusion_matrix[index, index])
        false_positive = float(confusion_matrix[:, index].sum() - true_positive)
        false_negative = float(confusion_matrix[index, :].sum() - true_positive)
        support = float(confusion_matrix[index, :].sum())
        precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
        recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({"label": class_name, "precision": precision, "recall": recall, "f1_score": f1, "support": int(support)})
    metrics = pd.DataFrame(rows)
    summary_rows = pd.DataFrame(
        [
            {"label": "accuracy", "precision": accuracy, "recall": accuracy, "f1_score": accuracy, "support": int(total)},
            {
                "label": "macro_avg",
                "precision": metrics["precision"].mean(),
                "recall": metrics["recall"].mean(),
                "f1_score": metrics["f1_score"].mean(),
                "support": int(total),
            },
            {
                "label": "weighted_avg",
                "precision": np.average(metrics["precision"], weights=metrics["support"]),
                "recall": np.average(metrics["recall"], weights=metrics["support"]),
                "f1_score": np.average(metrics["f1_score"], weights=metrics["support"]),
                "support": int(total),
            },
        ]
    )
    return pd.concat([metrics, summary_rows], ignore_index=True)

test_confusion_matrix = build_confusion_matrix(y_test, test_predictions, len(class_names))
test_metrics = classification_metrics_from_confusion(test_confusion_matrix, class_names)
for metric_column in ["precision", "recall", "f1_score"]:
    test_metrics[metric_column] = test_metrics[metric_column].round(4)

accuracy = float(test_metrics.loc[test_metrics["label"].eq("accuracy"), "f1_score"].iat[0])
macro_f1 = float(test_metrics.loc[test_metrics["label"].eq("macro_avg"), "f1_score"].iat[0])
weighted_f1 = float(test_metrics.loc[test_metrics["label"].eq("weighted_avg"), "f1_score"].iat[0])

print(f"Held-out test loss: {test_loss:.4f}")
print(f"Held-out test accuracy: {accuracy:.4f}")
print(f"Held-out test macro F1: {macro_f1:.4f}")
print(f"Held-out test weighted F1: {weighted_f1:.4f}")
display(test_metrics)

In [ ]:
# Build prediction artifacts and confusion matrix tables.
test_predictions_df = test_feature_rows.copy()
test_predictions_df["true_label_id"] = y_test
test_predictions_df["predicted_label_id"] = test_predictions
test_predictions_df["true_label"] = [id_to_label[int(label_id)] for label_id in y_test]
test_predictions_df["predicted_label"] = [id_to_label[int(label_id)] for label_id in test_predictions]
test_predictions_df["is_correct"] = test_predictions_df["true_label_id"].eq(test_predictions_df["predicted_label_id"])
for class_index, class_name in enumerate(class_names):
    safe_class_name = class_name.lower().replace(" ", "_")
    test_predictions_df[f"probability_{safe_class_name}"] = np.round(test_probabilities[:, class_index], 6)
test_predictions_df["predicted_probability"] = np.round(test_probabilities.max(axis=1), 6)

output_prediction_columns = [
    "phase6_row_id",
    "tweet_id",
    "entity",
    "model_text",
    "model_token_count",
    "true_label",
    "predicted_label",
    "predicted_probability",
    "probability_negative",
    "probability_neutral",
    "probability_positive",
    "is_correct",
]
test_predictions_df[output_prediction_columns].to_csv(PHASE8_FULL_PREDICTIONS_PATH, index=False)

test_confusion_matrix_df = pd.DataFrame(test_confusion_matrix, index=class_names, columns=class_names)
test_confusion_matrix_df.index.name = "actual_label"
test_confusion_matrix_df.to_csv(PHASE8_CONFUSION_MATRIX_PATH)

test_confusion_matrix_normalized = test_confusion_matrix_df.div(test_confusion_matrix_df.sum(axis=1), axis=0).fillna(0).round(4)
test_confusion_matrix_normalized.to_csv(PHASE8_CONFUSION_MATRIX_NORMALIZED_PATH)

test_metrics_with_loss = pd.concat(
    [
        test_metrics,
        pd.DataFrame([{"label": "weighted_cross_entropy_loss", "precision": test_loss, "recall": test_loss, "f1_score": test_loss, "support": len(y_test)}]),
    ],
    ignore_index=True,
)
test_metrics_with_loss.to_csv(PHASE8_TEST_METRICS_PATH, index=False)

prediction_distribution = (
    test_predictions_df.groupby(["true_label", "predicted_label"])
    .size()
    .rename("count")
    .reset_index()
)
prediction_distribution["percent_of_test"] = (prediction_distribution["count"] / len(test_predictions_df) * 100).round(2)
prediction_distribution.to_csv(PHASE8_PREDICTION_DISTRIBUTION_PATH, index=False)

correct_samples = test_predictions_df[test_predictions_df["is_correct"]].sample(
    min(8, int(test_predictions_df["is_correct"].sum())),
    random_state=42,
)
incorrect_samples = test_predictions_df[~test_predictions_df["is_correct"]].sample(
    min(12, int((~test_predictions_df["is_correct"]).sum())),
    random_state=42,
)
phase8_sample_predictions = pd.concat([correct_samples, incorrect_samples], ignore_index=True)
phase8_sample_predictions.insert(0, "sample_type", ["correct"] * len(correct_samples) + ["incorrect"] * len(incorrect_samples))
phase8_sample_predictions[["sample_type"] + output_prediction_columns].to_csv(PHASE8_SAMPLE_PREDICTIONS_PATH, index=False)

evaluation_overview = pd.DataFrame(
    {
        "metric": [
            "checkpoint_path",
            "best_validation_epoch",
            "best_validation_macro_f1",
            "test_rows",
            "test_loss_weighted_cross_entropy",
            "test_accuracy",
            "test_macro_f1",
            "test_weighted_f1",
            "test_split_used_for_training_or_tuning",
        ],
        "value": [
            str(PHASE7_CHECKPOINT_PATH),
            int(checkpoint["best_epoch"]),
            f"{float(checkpoint['best_validation_macro_f1']):.4f}",
            f"{len(y_test):,}",
            f"{test_loss:.4f}",
            f"{accuracy:.4f}",
            f"{macro_f1:.4f}",
            f"{weighted_f1:.4f}",
            "no",
        ],
    }
)
evaluation_overview.to_csv(PHASE8_EVALUATION_OVERVIEW_PATH, index=False)

print(f"Saved full test predictions to {PHASE8_FULL_PREDICTIONS_PATH}")
print(f"Saved test metrics to {PHASE8_TEST_METRICS_PATH}")
print(f"Saved test confusion matrix to {PHASE8_CONFUSION_MATRIX_PATH}")
display(test_confusion_matrix_df)
display(phase8_sample_predictions[["sample_type", "entity", "true_label", "predicted_label", "predicted_probability", "model_text"]].head(10))

In [ ]:
# Save an SVG confusion matrix for the held-out test evaluation.
def svg_text(text):
    import html
    return html.escape(str(text), quote=True)


def save_confusion_matrix_svg(confusion_matrix_df, normalized_df, path, width=760, height=620):
    labels = list(confusion_matrix_df.index)
    matrix_size = 360
    cell_size = matrix_size / len(labels)
    margin_left, margin_top = 190, 115
    max_count = max(int(confusion_matrix_df.to_numpy().max()), 1)
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="38" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Phase 8 Test Confusion Matrix</text>',
        f'<text x="{margin_left + matrix_size/2}" y="{height-35}" text-anchor="middle" font-family="Arial" font-size="14" fill="#333">Predicted label</text>',
        f'<text x="35" y="{margin_top + matrix_size/2}" text-anchor="middle" transform="rotate(-90 35 {margin_top + matrix_size/2})" font-family="Arial" font-size="14" fill="#333">Actual label</text>',
    ]
    for column_index, predicted_label in enumerate(labels):
        x = margin_left + column_index * cell_size + cell_size / 2
        parts.append(f'<text x="{x:.1f}" y="{margin_top-18}" text-anchor="middle" font-family="Arial" font-size="14" font-weight="700" fill="#222">{svg_text(predicted_label)}</text>')
    for row_index, actual_label in enumerate(labels):
        y = margin_top + row_index * cell_size + cell_size / 2
        parts.append(f'<text x="{margin_left-18}" y="{y+5:.1f}" text-anchor="end" font-family="Arial" font-size="14" font-weight="700" fill="#222">{svg_text(actual_label)}</text>')
        for column_index, predicted_label in enumerate(labels):
            count = int(confusion_matrix_df.loc[actual_label, predicted_label])
            rate = float(normalized_df.loc[actual_label, predicted_label])
            intensity = count / max_count
            blue = int(245 - intensity * 120)
            green = int(248 - intensity * 105)
            red = int(242 - intensity * 90)
            fill = f"rgb({red},{green},{blue})"
            x = margin_left + column_index * cell_size
            y_cell = margin_top + row_index * cell_size
            text_color = "#ffffff" if intensity > 0.55 else "#1f2933"
            parts.extend([
                f'<rect x="{x:.1f}" y="{y_cell:.1f}" width="{cell_size:.1f}" height="{cell_size:.1f}" fill="{fill}" stroke="#ffffff" stroke-width="3"/>',
                f'<text x="{x + cell_size/2:.1f}" y="{y_cell + cell_size/2 - 6:.1f}" text-anchor="middle" font-family="Arial" font-size="22" font-weight="700" fill="{text_color}">{count:,}</text>',
                f'<text x="{x + cell_size/2:.1f}" y="{y_cell + cell_size/2 + 20:.1f}" text-anchor="middle" font-family="Arial" font-size="13" fill="{text_color}">{rate*100:.1f}%</text>',
            ])
    path.write_text(
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" role="img">\n'
        + "\n".join(parts)
        + "\n</svg>\n",
        encoding="utf-8",
    )

save_confusion_matrix_svg(test_confusion_matrix_df, test_confusion_matrix_normalized, PHASE8_CONFUSION_MATRIX_FIGURE_PATH)
print(f"Saved test confusion matrix figure to {PHASE8_CONFUSION_MATRIX_FIGURE_PATH}")

In [ ]:
# Save concise Phase 8 evaluation report.
negative_f1 = float(test_metrics.loc[test_metrics["label"].eq("Negative"), "f1_score"].iat[0])
neutral_f1 = float(test_metrics.loc[test_metrics["label"].eq("Neutral"), "f1_score"].iat[0])
positive_f1 = float(test_metrics.loc[test_metrics["label"].eq("Positive"), "f1_score"].iat[0])
most_common_error = (
    prediction_distribution[prediction_distribution["true_label"].ne(prediction_distribution["predicted_label"])]
    .sort_values("count", ascending=False)
    .head(1)
)
if most_common_error.empty:
    most_common_error_text = "None"
else:
    error_row = most_common_error.iloc[0]
    most_common_error_text = f"{error_row['true_label']} predicted as {error_row['predicted_label']} ({int(error_row['count']):,} rows)"

phase8_summary_lines = [
    "# Phase 8 Held-Out Test Evaluation Summary",
    "",
    f"- Evaluated checkpoint: {PHASE7_CHECKPOINT_PATH}",
    f"- Full test predictions: {PHASE8_FULL_PREDICTIONS_PATH}",
    f"- Test metrics: {PHASE8_TEST_METRICS_PATH}",
    f"- Test confusion matrix: {PHASE8_CONFUSION_MATRIX_PATH}",
    f"- Normalized test confusion matrix: {PHASE8_CONFUSION_MATRIX_NORMALIZED_PATH}",
    f"- Test prediction samples: {PHASE8_SAMPLE_PREDICTIONS_PATH}",
    f"- Confusion matrix figure: {PHASE8_CONFUSION_MATRIX_FIGURE_PATH}",
    f"- Test rows evaluated: {len(y_test):,}",
    f"- Test weighted cross-entropy loss: {test_loss:.4f}",
    f"- Test accuracy: {accuracy:.4f}",
    f"- Test macro F1: {macro_f1:.4f}",
    f"- Test weighted F1: {weighted_f1:.4f}",
    f"- Negative F1: {negative_f1:.4f}",
    f"- Neutral F1: {neutral_f1:.4f}",
    f"- Positive F1: {positive_f1:.4f}",
    f"- Most common error pattern: {most_common_error_text}",
    "- Model retraining or tuning in this phase: no.",
    "",
    "## Carry-Forward Notes",
    "",
    "- Phase 9 should use these held-out test results as the baseline for improvement comparisons.",
    "- Neutral is the weakest class for this baseline and should be watched during improvement experiments.",
    "- Future experiments should tune on validation data first and only compare against this test result after selecting a final candidate.",
]
PHASE8_SUMMARY_PATH.write_text("\n".join(phase8_summary_lines) + "\n", encoding="utf-8")

print(f"Saved Phase 8 summary to {PHASE8_SUMMARY_PATH}")

## Phase 8 Findings To Carry Forward

Phase 8 evaluates the baseline GRU once on the held-out test split.

Carry-forward points:

- Treat the Phase 8 test metrics as the baseline held-out performance for the project.
- Use validation data, not the test split, when exploring improvements in Phase 9.
- Compare any improved final model against this baseline only after selecting it from validation results.
- Pay special attention to neutral-class performance, which is the weakest class for the baseline.

## Phase 9: Model Improvement

Phase 9 compares a small set of validation-driven RNN variants against the Phase 7 baseline.

Work completed in this phase:

- Load the Phase 6 train and validation splits.
- Use the Phase 7 validation metrics as the baseline reference.
- Train a small set of GRU/LSTM candidate variants using train data only.
- Select the best candidate by validation macro F1.
- Save experiment comparisons, best validation metrics, model metadata, and improvement figures.

The held-out test split is not loaded or evaluated in this phase. Phase 8 remains the held-out baseline result.

In [ ]:
# Phase 9 implementation lives in src so the experiment can be rerun outside the notebook as well.
import runpy
from pathlib import Path

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

phase9_script_path = PROJECT_ROOT / "src" / "phase9_model_improvement.py"
phase9_result = runpy.run_path(str(phase9_script_path), run_name="__main__")
print(f"Executed Phase 9 script: {phase9_script_path}")

## Phase 9 Findings To Carry Forward

Phase 9 compares validation-driven RNN improvements against the Phase 7 baseline.

Carry-forward points:

- Use `outputs/tables/phase9_experiment_results.csv` as the model-improvement comparison table.
- Use `outputs/models/phase9_best_validation_model_state.pt` only as a validation-selected improved checkpoint.
- Keep Phase 8's held-out test result separate from Phase 9 validation-selection results.
- Phase 10 should present the baseline test result, the validation improvement attempt, and any limitations clearly.

## Phase 10: Final Report and Presentation

Phase 10 packages the completed project for submission. It does not run training, tuning, or new held-out test evaluation.

Work completed in this phase:

- Generate the final project report from saved Phase 2-9 artifacts.
- Consolidate headline metrics into a final key-metrics table.
- Create a premium executive presentation deck for final delivery.
- Keep Phase 8 held-out test metrics and Phase 9 validation improvements clearly separated.


In [ ]:
# Phase 10 packages existing artifacts only; it does not train or evaluate a model.
import runpy
from pathlib import Path

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

runpy.run_path(str(PROJECT_ROOT / "src" / "phase10_final_deliverables.py"), run_name="__main__")


## Phase 10 Final Deliverables

Final submission artifacts:

- Final report: `outputs/reports/final_project_report.md`
- Final presentation: `outputs/reports/rnn_twitter_sentiment_final_presentation.pptx`
- Final key metrics: `outputs/tables/phase10_final_key_metrics.csv`
- Submission checklist: `outputs/reports/phase10_final_submission_checklist.md`

Final reporting boundary:

- Phase 8 is the official held-out test baseline: accuracy `0.6387`, macro F1 `0.6262`.
- Phase 9 is validation-only model selection: best validation macro F1 `0.6874`.
- If another modeling step is permitted, run exactly one final held-out test evaluation for the selected Phase 9 checkpoint.
